In [5]:
from pathlib import Path
from collections import namedtuple
import pandas as pd
import numpy as np
from numpy import random
import shutil
import os
import sys
import traceback
import re
from pydub import AudioSegment
from pydub.utils import make_chunks

from datetime import datetime
import librosa
import soundfile as sf

In [7]:
# =============================
# BASE WORKING DIRS
# ============================
# Speech data folder in Google Drive
DIR_DATA_GOOGLE_DRIVE = Path("/Users/dmatekenya/Library/CloudStorage/GoogleDrive-dmatekenya@gmail.com/My Drive/Chichewa-ASR2/data/chichewa-speech-data/")
DIR_GOOGLE_SUBMISSIONS = Path("/Users/dmatekenya/Library/CloudStorage/GoogleDrive-dmatekenya@gmail.com/My Drive/Chichewa-ASR2/data/chichewa-google-submission/chich_speech_dataset/chich_speech_audio_files")
DIR_AUDIO_FILES = DIR_DATA_GOOGLE_DRIVE / "chich-audio-transcript-files-v1"
print(DIR_AUDIO_FILES.exists())




# Data folder in this repository
DIR_DATA = Path.cwd().parents[0].joinpath('data')

DIR_FEMALE_AUDIO = DIR_DATA_GOOGLE_DRIVE / "dev_test_aims_cameroon/female"
assert DIR_FEMALE_AUDIO.exists()
OUT_DIR = DIR_DATA_GOOGLE_DRIVE / "aims-cameroon/test-females"
# assert OUT_DIR.exists()


True


In [21]:
# ==========================================================
# CALCULATE TOTAL DURATION OF AUDIO FILES IN THE DATASET
# ========================================================== 
total_duration_sec = 0

for file in DIR_AUDIO_FILES.iterdir():
    if file.suffix == ".wav":
        audio = AudioSegment.from_file(file)
        total_duration_sec += len(audio) // 1000  # duration in seconds

print(f"Total duration: {total_duration_sec/3600:.2f} hours ({total_duration_sec} seconds)")

Total duration: 23.45 hours (84430 seconds)


In [22]:
# Define output directory for the 5-hour sample
sample_output_dir = DIR_DATA_GOOGLE_DRIVE / "sample_5hr"
sample_output_dir.mkdir(parents=True, exist_ok=True)

sampled_duration = 0
target_duration = 5 * 3600  # 5 hours in seconds

for audio_file in DIR_AUDIO_FILES.iterdir():
    if audio_file.suffix == ".wav":
        audio = AudioSegment.from_file(audio_file)
        duration = len(audio) // 1000  # duration in seconds
        if sampled_duration + duration > target_duration:
            # Only take the needed portion to reach 5 hours
            remaining = target_duration - sampled_duration
            sample_audio = audio[:remaining * 1000]
            sample_audio.export(sample_output_dir / audio_file.name, format="wav")
            # Copy transcript if exists
            transcript_file = audio_file.with_suffix('.txt')
            if transcript_file.exists():
                shutil.copy(transcript_file, sample_output_dir / transcript_file.name)
            break
        else:
            # Copy full audio
            shutil.copy(audio_file, sample_output_dir / audio_file.name)
            # Copy transcript if exists
            transcript_file = audio_file.with_suffix('.txt')
            if transcript_file.exists():
                shutil.copy(transcript_file, sample_output_dir / transcript_file.name)
            sampled_duration += duration

print(f"Sampled {sampled_duration/3600:.2f} hours of audio to {sample_output_dir}")

Sampled 4.89 hours of audio to /Users/dmatekenya/Library/CloudStorage/GoogleDrive-dmatekenya@gmail.com/My Drive/Chichewa-ASR2/data/chichewa-speech-data/sample_5hr


In [38]:

# =========================================================
# CONFIG
# =========================================================
AUDIO_EXTENSIONS = {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".opus", ".aac"}

BUCKET_TARGETS_MIN = {
    "1_5": 5.0,
    "5_10": 20,
    "10_15": 15,
    "15_20": 12.5,
    "20_30": 12.5,
    "gt_30": None,  # take all
}
RANDOM_STATE = 42


# =========================================================
# HELPERS
# =========================================================
def get_audio_duration_seconds(file_path: Path) -> float:
    info = sf.info(file_path)
    return float(info.frames) / float(info.samplerate)


def assign_duration_bucket(duration_seconds: float) -> str | None:
    if duration_seconds < 1:
        return None
    if 1 <= duration_seconds < 5:
        return "1_5"
    if 5 <= duration_seconds < 10:
        return "5_10"
    if 10 <= duration_seconds < 15:
        return "10_15"
    if 15 <= duration_seconds < 20:
        return "15_20"
    if 20 <= duration_seconds <= 30:
        return "20_30"
    if duration_seconds > 30:
        return "gt_30"
    return None


def build_audio_inventory(audio_dir: Path) -> pd.DataFrame:
    rows = []

    for file_path in sorted(audio_dir.rglob("*")):
        if not file_path.is_file():
            continue
        if file_path.suffix.lower() not in AUDIO_EXTENSIONS:
            continue

        try:
            duration_seconds = get_audio_duration_seconds(file_path)
        except Exception as e:
            print(f"Skipping {file_path.name}: {e}")
            continue

        bucket = assign_duration_bucket(duration_seconds)

        rows.append(
            {
                "file_path": str(file_path),
                "file_name": file_path.name,
                "duration_seconds": duration_seconds,
                "duration_minutes": duration_seconds / 60.0,
                "duration_bucket": bucket,
            }
        )

    df = pd.DataFrame(rows)
    return df


def sample_bucket(df_bucket: pd.DataFrame, target_minutes: float | None, random_state: int) -> pd.DataFrame:
    if df_bucket.empty:
        return df_bucket.copy()

    total_available = df_bucket["duration_minutes"].sum()

    # Take all if:
    # - no target (gt_30)
    # - or not enough audio
    if target_minutes is None or total_available <= target_minutes:
        return df_bucket.copy().reset_index(drop=True)

    # Otherwise sample randomly until target reached
    df_bucket = df_bucket.sample(frac=1, random_state=random_state).reset_index(drop=True)

    selected = []
    total = 0.0

    for _, row in df_bucket.iterrows():
        if total >= target_minutes:
            break
        selected.append(row)
        total += row["duration_minutes"]

    return pd.DataFrame(selected)


def copy_files(sampled_df: pd.DataFrame, out_dir: Path) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)

    for _, row in sampled_df.iterrows():
        src = Path(row["file_path"])
        dst = out_dir / src.name

        # Handle duplicate names safely
        counter = 1
        while dst.exists():
            dst = out_dir / f"{src.stem}_{counter}{src.suffix}"
            counter += 1

        shutil.copy2(src, dst)


# =========================================================
# MAIN
# =========================================================
def main() -> None:
    df = build_audio_inventory(DIR_FEMALE_AUDIO)

    # Keep only valid buckets
    df = df[df["duration_bucket"].notna()].copy()

    sampled_parts = []

    for i, (bucket, target) in enumerate(BUCKET_TARGETS_MIN.items()):
        df_bucket = df[df["duration_bucket"] == bucket]

        sampled = sample_bucket(
            df_bucket=df_bucket,
            target_minutes=target,
            random_state=RANDOM_STATE + i,
        )

        sampled_parts.append(sampled)

        print(
            f"{bucket}: available={df_bucket['duration_minutes'].sum():.2f} min | "
            f"target={target} | sampled={sampled['duration_minutes'].sum():.2f} min"
        )

    sampled_df = pd.concat(sampled_parts, ignore_index=True)

    # Save metadata
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    sampled_df.to_csv(OUT_DIR / "female_sample_metadata.csv", index=False)

    # Copy files (flat structure)
    copy_files(sampled_df, OUT_DIR)

    print("\n=== FINAL SUMMARY ===")
    print(sampled_df.groupby("duration_bucket")["duration_minutes"].sum())
    print(f"\nTotal minutes: {sampled_df['duration_minutes'].sum():.2f}")
    print(f"Total files: {len(sampled_df)}")




In [42]:
dev_test_dir = DIR_DATA_GOOGLE_DRIVE / "dev_test_aims_cameroon"
file_transcripts = dev_test_dir / "metadata_transcripts.csv"

In [45]:
# Build test_females_metadata by adding transcript text to existing OUT_DIR metadata

metadata_path = OUT_DIR / "female_sample_metadata.csv"

# 1) Load base metadata (or build from OUT_DIR audio files if metadata file is missing)
if metadata_path.exists():
    df_meta = pd.read_csv(metadata_path)
else:
    audio_files = [
        p for p in OUT_DIR.iterdir()
        if p.is_file() and p.suffix.lower() in AUDIO_EXTENSIONS
    ]
    df_meta = pd.DataFrame({"audio_filename": [p.name for p in sorted(audio_files)]})

# Normalize audio filename column name
if "audio_filename" not in df_meta.columns:
    if "file_name" in df_meta.columns:
        df_meta["audio_filename"] = df_meta["file_name"]
    elif "filename" in df_meta.columns:
        df_meta["audio_filename"] = df_meta["filename"]
    else:
        raise ValueError("Could not find an audio filename column in metadata.")

# 2) Build transcript lookup from dev_test_dir .txt files
txt_lookup = {}  # key: lowercase stem -> Path
for txt_path in dev_test_dir.rglob("*.txt"):
    txt_lookup[txt_path.stem.lower()] = txt_path

def safe_read_text(path_obj: Path):
    try:
        return path_obj.read_text(encoding="utf-8", errors="replace").strip()
    except Exception:
        return np.nan

# 3) Build fallback lookup from file_transcripts CSV
def _norm_col(c: str) -> str:
    return str(c).strip().lower().replace("-", "_").replace(" ", "_")

csv_lookup = {}  # key: lowercase id/name/stem -> transcript text
if file_transcripts.exists():
    df_trans = pd.read_csv(file_transcripts)

    id_candidates = {
        "audio_filename", "file_name", "filename", "audio", "audio_file", "id", "file_id"
    }
    text_candidates = {
        "transcript", "transcription", "text", "sentence", "utterance",
        "chichewa_transcript", "chichewa-transcript"
    }

    norm_to_orig = {_norm_col(c): c for c in df_trans.columns}

    id_col = next((norm_to_orig[c] for c in id_candidates if c in norm_to_orig), None)
    text_col = next((norm_to_orig[c] for c in text_candidates if c in norm_to_orig), None)

    if id_col is not None and text_col is not None:
        for _, r in df_trans[[id_col, text_col]].dropna().iterrows():
            raw_id = str(r[id_col]).strip()
            text_val = str(r[text_col]).strip()

            # store multiple useful keys
            csv_lookup[raw_id.lower()] = text_val
            csv_lookup[Path(raw_id).stem.lower()] = text_val
    else:
        print("Warning: Could not infer ID/text columns in file_transcripts CSV.")
        print("Columns:", list(df_trans.columns))
else:
    print(f"Warning: file_transcripts does not exist -> {file_transcripts}")

# 4) Attach transcript to metadata
transcript_filenames = []
transcript_texts = []
transcript_sources = []

for audio_name in df_meta["audio_filename"].astype(str):
    stem = Path(audio_name).stem.lower()

    # Priority 1: transcript text file in dev_test_dir
    if stem in txt_lookup:
        txt_path = txt_lookup[stem]
        transcript_filenames.append(txt_path.name)
        transcript_texts.append(safe_read_text(txt_path))
        transcript_sources.append("dev_test_dir_txt")
        continue

    # Priority 2: fallback CSV (file_transcripts)
    txt_from_csv = csv_lookup.get(audio_name.lower()) or csv_lookup.get(stem)
    if txt_from_csv is not None:
        transcript_filenames.append(np.nan)
        transcript_texts.append(txt_from_csv)
        transcript_sources.append("metadata_transcripts_csv")
    else:
        transcript_filenames.append(np.nan)
        transcript_texts.append(np.nan)
        transcript_sources.append("missing")

df_meta["transcript_filename"] = transcript_filenames
df_meta["transcript"] = transcript_texts
df_meta["transcript_source"] = transcript_sources

# 5) Save output
test_females_metadata_path = OUT_DIR / "test_females_metadata.csv"
df_meta.to_csv(test_females_metadata_path, index=False)

print(f"Saved: {test_females_metadata_path}")
print(df_meta["transcript_source"].value_counts(dropna=False))
display(df_meta.head(10))

Saved: /Users/dmatekenya/Library/CloudStorage/GoogleDrive-dmatekenya@gmail.com/My Drive/Chichewa-ASR2/data/chichewa-speech-data/aims-cameroon/test-females/test_females_metadata.csv
transcript_source
dev_test_dir_txt            321
metadata_transcripts_csv      7
missing                       1
Name: count, dtype: int64


,file_path,file_name,duration_seconds,duration_minutes,duration_bucket,audio_filename,transcript_filename,transcript,transcript_source
0,/Users/dmatekenya/Library/CloudStorage/GoogleD...,priscilla_self_records46.wav,4.388571,0.073143,1_5,priscilla_self_records46.wav,priscilla_self_records46.txt,Kodi zikhadabo amaika bwanji amene akudziwa an...,dev_test_dir_txt
1,/Users/dmatekenya/Library/CloudStorage/GoogleD...,priscilla_self_records414.wav,4.458231,0.074304,1_5,priscilla_self_records414.wav,priscilla_self_records414.txt,Akuti adavulala kwambiri anavulallira mkati,dev_test_dir_txt
2,/Users/dmatekenya/Library/CloudStorage/GoogleD...,priscilla_self_records418.wav,3.715193,0.061920,1_5,priscilla_self_records418.wav,priscilla_self_records418.txt,Mtsikana uyu amaphoda kwambiri,dev_test_dir_txt
3,/Users/dmatekenya/Library/CloudStorage/GoogleD...,priscilla_self_records211.wav,4.690431,0.078174,1_5,priscilla_self_records211.wav,priscilla_self_records211.txt,Kaodeni katundu ku Kampepuza ku Ntcheu.,dev_test_dir_txt
4,/Users/dmatekenya/Library/CloudStorage/GoogleD...,AU40.wav,2.560000,0.042667,1_5,AU40.wav,AU40.txt,Tazimitsako magetsiwo Tazimitsako magetsiwo,dev_test_dir_txt
5,/Users/dmatekenya/Library/CloudStorage/GoogleD...,priscilla_self_records114.wav,4.876190,0.081270,1_5,priscilla_self_records114.wav,priscilla_self_records114.txt,Kodi nkhani zako za dzulo zija zakuthera bwanji.,dev_test_dir_txt
6,/Users/dmatekenya/Library/CloudStorage/GoogleD...,priscilla_self_records575.wav,3.900952,0.065016,1_5,priscilla_self_records575.wav,priscilla_self_records575.txt,Ma windowa ndi akuda akuyenera kuchapidwa.,dev_test_dir_txt
7,/Users/dmatekenya/Library/CloudStorage/GoogleD...,AU98.wav,2.925714,0.048762,1_5,AU98.wav,AU98.txt,Ku ntchito sindipitako ndadwala.Ku ntchito sin...,dev_test_dir_txt
8,/Users/dmatekenya/Library/CloudStorage/GoogleD...,priscilla_self_records7.wav,4.690431,0.078174,1_5,priscilla_self_records7.wav,priscilla_self_records7.txt,Funso limenelo anafunsa kale Grace amuthandiza,dev_test_dir_txt
9,/Users/dmatekenya/Library/CloudStorage/GoogleD...,priscilla_self_records218.wav,3.970612,0.066177,1_5,priscilla_self_records218.wav,priscilla_self_records218.txt,Akuti ayi sanazipeze,dev_test_dir_txt


In [48]:
df_meta.dropna(subset=["transcript"], inplace=True)

In [51]:
df_meta = df_meta[['file_name', 'audio_filename', 'duration_seconds', 'duration_minutes',
       'duration_bucket', 'transcript_filename',
       'transcript']]

In [ ]:
df_meta = df_meta[['audio_filename', 'transcript_filename', 'duration_seconds', 'transcript']]
df_meta

,audio_filename,transcript_filename,duration_seconds,transcript
0,priscilla_self_records46.wav,priscilla_self_records46.txt,4.388571,Kodi zikhadabo amaika bwanji amene akudziwa an...
1,priscilla_self_records414.wav,priscilla_self_records414.txt,4.458231,Akuti adavulala kwambiri anavulallira mkati
2,priscilla_self_records418.wav,priscilla_self_records418.txt,3.715193,Mtsikana uyu amaphoda kwambiri
3,priscilla_self_records211.wav,priscilla_self_records211.txt,4.690431,Kaodeni katundu ku Kampepuza ku Ntcheu.
4,AU40.wav,AU40.txt,2.560000,Tazimitsako magetsiwo Tazimitsako magetsiwo
...,...,...,...,...
323,priscilla_self_records182.wav,priscilla_self_records182.txt,21.524898,Eee baba tikuoneni nyali yazimayi walitseni; e...
324,priscilla_self_records28.wav,priscilla_self_records28.txt,21.733878,Zikanakhala bwino zikanakhala kuti pali munthu...
325,priscilla_self_records304.wav,priscilla_self_records304.txt,24.218413,Lidali tsiku la bwino kwambir pomwe ndidaona z...
326,priscilla_self_records57.wav,priscilla_self_records57.txt,21.989297,Komabe sikuti ndili sure kuti ku dabako ndi Ch...


In [80]:
def extract_speaker_id(filename: str) -> str:
    filename = Path(filename).stem.lower()

    # 1. Known names
    if "priscilla" in filename:
        return "priscilla"

    # 2. Generic audio files
    if re.match(r"^audio[0-9]*$", filename):
        return "audio_speaker"

    # 3. WAU prefix
    if filename.startswith("wau"):
        return "wau_speaker"

    # 4. AU prefix, but not "audio"
    if re.match(r"^au($|[_-]|\d)", filename):
        return "au_speaker"

    # 5. Remove numeric suffix (_1, _2, etc.)
    root = re.sub(r"_[0-9]+$", "", filename)

    return root

In [81]:
# Apply to your dataframe
df_meta["speaker_id"] = df_meta["audio_filename"].apply(extract_speaker_id)

In [82]:
df_meta.query('audio_filename.str.contains("^audio", case=False, regex=True)')

,audio_filename,transcript_filename,duration_seconds,transcript,speaker_id
104,audio10.mp3,NaN,7.304807,Eya..ndibwino kupalila mmunda wathu kuti mbeu ...,audio_speaker
142,audio4.mp3,NaN,9.896893,Ubwino osunga ma rekodi oyamba ndiwokuti; zima...,audio_speaker
152,audio5.mp3,NaN,8.724014,Zimatithandizilanso kuti tione kuti kodi pa mu...,audio_speaker
239,audio3.mp3,NaN,7.365760,"kachitatu, timadzala mtedza chifukwa masamba a...",audio_speaker
259,audio1.mp3,NaN,10.158730,Timadzala mtedza chifukwa chiyamba ndi mmene t...,audio_speaker
260,audio16.mp3,NaN,11.939138,Mbeu zimene ali ambiri amakonda kulima mmalawi...,audio_speaker
261,audio2.mp3,NaN,10.709184,Chachiwiri timadzala mtedza chifukwa chonena k...,audio_speaker


In [85]:
df_meta.to_csv(OUT_DIR / "test_females_metadata.csv", index=False)

In [16]:
# =========================================================
# HELPERS
# =========================================================
def get_audio_duration_seconds(file_path: Path) -> float:
    """
    Get audio duration in seconds using soundfile.

    Parameters
    ----------
    file_path : Path
        Path to the audio file.

    Returns
    -------
    float
        Duration in seconds.
    """
    info = sf.info(file_path)
    return float(info.frames) / float(info.samplerate)


In [27]:
def assign_duration_bucket(duration_seconds: float) -> str | None:
    """
    Assign a non-overlapping duration bucket.

    Parameters
    ----------
    duration_seconds : float
        Audio duration in seconds.

    Returns
    -------
    str | None
        Duration bucket label, or None if outside 1-30 seconds.
    """
    if duration_seconds < 1:
        return None
    if 1 <= duration_seconds < 5:
        return "1_5"
    if 5 <= duration_seconds < 10:
        return "5_10"
    if 10 <= duration_seconds < 15:
        return "10_15"
    if 15 <= duration_seconds < 20:
        return "15_20"
    if 20 <= duration_seconds <= 40:
        return "20_30"
    return None

In [28]:
def build_audio_inventory(audio_dir: Path) -> pd.DataFrame:
    """
    Scan folder and build dataframe with durations and buckets.

    Parameters
    ----------
    audio_dir : Path
        Folder containing audio clips.

    Returns
    -------
    pd.DataFrame
        Inventory dataframe.
    """
    rows = []

    for file_path in sorted(audio_dir.rglob("*")):
        if not file_path.is_file():
            continue
        if file_path.suffix.lower() not in AUDIO_EXTENSIONS:
            continue

        try:
            duration_seconds = get_audio_duration_seconds(file_path)
        except Exception as e:
            print(f"Skipping {file_path.name}: {e}")
            continue

        bucket = assign_duration_bucket(duration_seconds)

        rows.append(
            {
                "file_path": str(file_path),
                "file_name": file_path.name,
                "duration_seconds": duration_seconds,
                "duration_minutes": duration_seconds / 60.0,
                "duration_bucket": bucket,
            }
        )

    df = pd.DataFrame(rows)

    if df.empty:
        raise ValueError("No readable audio files found in the input folder.")

    return df

In [19]:
def copy_sampled_files(sampled_df: pd.DataFrame, out_dir: Path) -> None:
    """
    Copy sampled files into bucket subfolders in out_dir.

    Parameters
    ----------
    sampled_df : pd.DataFrame
        Sampled audio dataframe.
    out_dir : Path
        Output directory.
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    for _, row in sampled_df.iterrows():
        src = Path(row["file_path"])
        bucket = row["duration_bucket"]
        dst_dir = out_dir / bucket
        dst_dir.mkdir(parents=True, exist_ok=True)

        dst = dst_dir / src.name
        shutil.copy2(src, dst)


In [20]:
def sample_bucket(
    df_bucket: pd.DataFrame,
    target_minutes: float,
    random_state: int,
) -> pd.DataFrame:
    """
    Sample clips within a bucket until target duration is reached or exceeded.

    Parameters
    ----------
    df_bucket : pd.DataFrame
        Dataframe already filtered to one duration bucket.
    target_minutes : float
        Target minutes for that bucket.
    random_state : int
        Random seed.

    Returns
    -------
    pd.DataFrame
        Sampled rows.
    """
    if df_bucket.empty:
        return df_bucket.copy()

    # Shuffle for reproducibility, then greedily accumulate
    df_bucket = df_bucket.sample(frac=1, random_state=random_state).reset_index(drop=True)

    selected_rows = []
    total_minutes = 0.0

    for _, row in df_bucket.iterrows():
        if total_minutes >= target_minutes:
            break
        selected_rows.append(row)
        total_minutes += row["duration_minutes"]

    if not selected_rows:
        return df_bucket.iloc[0:0].copy()

    sampled_df = pd.DataFrame(selected_rows)
    return sampled_df

In [ ]:

AUDIO_EXTENSIONS = {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".opus", ".aac"}

# Target duration per bucket in minutes
BUCKET_TARGETS_MIN = {
    "1_5": 5,
    "5_10": 10,
    "10_15": 6,
    "15_20": 10,
    "20_30": 10,
}

RANDOM_STATE = 42

OUT_DIR.mkdir(parents=True, exist_ok=True)

df = build_audio_inventory(DIR_FEMALE_AUDIO)

# Keep only 1-30 second clips
df = df[df["duration_bucket"].notna()].copy()

sampled_parts = []

for i, (bucket, target_minutes) in enumerate(BUCKET_TARGETS_MIN.items(), start=1):
    df_bucket = df[df["duration_bucket"] == bucket].copy()

    sampled_bucket = sample_bucket(
        df_bucket=df_bucket,
        target_minutes=target_minutes,
        random_state=RANDOM_STATE + i,
    )

    sampled_parts.append(sampled_bucket)

    available_minutes = df_bucket["duration_minutes"].sum()
    sampled_minutes = sampled_bucket["duration_minutes"].sum()

    print(
        f"{bucket}: "
        f"available={available_minutes:.2f} min, "
        f"target={target_minutes:.2f} min, "
        f"sampled={sampled_minutes:.2f} min, "
        f"n_files={len(sampled_bucket)}"
    )

sampled_df = pd.concat(sampled_parts, ignore_index=True)

# Save metadata
sampled_df.to_csv(OUT_DIR / "female_sample_metadata.csv", index=False)

# Copy files
copy_sampled_files(sampled_df, OUT_DIR)

# Summary
print("\n=== FINAL SUMMARY ===")
print(sampled_df.groupby("duration_bucket")["duration_minutes"].sum())
print(f"\nTotal sampled minutes: {sampled_df['duration_minutes'].sum():.2f}")

1_5: available=23.91 min, target=4.00 min, sampled=4.02 min, n_files=63
5_10: available=32.07 min, target=6.00 min, sampled=6.06 min, n_files=53
10_15: available=9.50 min, target=6.00 min, sampled=6.21 min, n_files=31
15_20: available=1.68 min, target=10.00 min, sampled=1.68 min, n_files=6
20_30: available=6.56 min, target=10.00 min, sampled=6.56 min, n_files=16

=== FINAL SUMMARY ===
duration_bucket
10_15    6.209016
15_20    1.684743
1_5      4.017577
20_30    6.556393
5_10     6.060369
Name: duration_minutes, dtype: float64

Total sampled minutes: 24.53


## Find Missing Audio and Transcript Files
Read `missing_files.csv` and search recursively across all folders in the speech data directory.

In [14]:
# Step 1: Read missing_files.csv
df_missing = pd.read_csv(DIR_DATA.joinpath("missing_files.csv"))
print(f"Missing audio entries: {df_missing['no_audio'].dropna().shape[0]}")
print(f"Missing transcript entries: {df_missing['no_transcript'].dropna().shape[0]}")
df_missing.head()

Missing audio entries: 115
Missing transcript entries: 113


,no_audio,no_transcript
0,220422-162004_nya_e46,220422-123447_nya_e46
1,220422-180143_nya_e46 (1),220422-151734_nya_e46
2,220423-105315_nya_e46,220423-103230_nya_e46
3,220423-115434_nya_e46,220423-114718_nya_e46 (1)
4,220423-123824_nya_e46,220423-121056_nya_e46


In [18]:
# Step 2: Recursively scan all folders to build a file index
# Maps base filename (without extension) -> list of full paths
from collections import defaultdict

AUDIO_EXTENSIONS = {'.wav', '.mp3', '.m4a', '.aac', '.ogg', '.flac', '.mp4', '.opus'}

file_index_audio = defaultdict(list)  # base_name -> [full_paths] (any audio format)
file_index_txt = defaultdict(list)    # base_name -> [full_paths]

# Output folder for recovered files
DIR_RECOVERED = DIR_DATA_GOOGLE_DRIVE / "recovered_missing_files"
DIR_RECOVERED.mkdir(parents=True, exist_ok=True)

scan_root = DIR_DATA_GOOGLE_DRIVE

print(f"Scanning: {scan_root}")
print(f"Directory exists: {scan_root.exists()}")
print(f"Recovered files will be copied to: {DIR_RECOVERED}")

for fpath in scan_root.rglob("*"):
    if fpath.is_file():
        base_name = fpath.stem  # filename without extension

        if fpath.suffix.lower() in AUDIO_EXTENSIONS:
            file_index_audio[base_name].append(fpath)
        elif fpath.suffix.lower() == ".txt":
            file_index_txt[base_name].append(fpath)

print(f"Indexed {len(file_index_audio)} unique audio base names")
print(f"Indexed {len(file_index_txt)} unique .txt base names")


Scanning: /Users/dmatekenya/Library/CloudStorage/GoogleDrive-dmatekenya@gmail.com/My Drive/Chichewa-ASR2/data/chichewa-speech-data
Directory exists: True
Recovered files will be copied to: /Users/dmatekenya/Library/CloudStorage/GoogleDrive-dmatekenya@gmail.com/My Drive/Chichewa-ASR2/data/chichewa-speech-data/recovered_missing_files
Indexed 19924 unique audio base names
Indexed 4737 unique .txt base names


In [20]:
# Step 3: Search for missing audio files (any audio format) and copy found ones
missing_audio_ids = df_missing['no_audio'].dropna().tolist()

audio_found = []
audio_not_found = []
audio_copied = 0

for file_id in missing_audio_ids:
    file_id = str(file_id).strip()
    found = False
    # Direct match
    if file_id in file_index_audio:
        for found_path in file_index_audio[file_id]:
            audio_found.append({"missing_id": file_id, "found_path": str(found_path)})
            dest = DIR_RECOVERED / found_path.name
            if not dest.exists():
                shutil.copy2(found_path, dest)
                audio_copied += 1
        found = True
    else:
        # Try without " (1)" suffix in case it's a duplicate marker
        clean_id = re.sub(r'\s*\(\d+\)$', '', file_id)
        if clean_id != file_id and clean_id in file_index_audio:
            for found_path in file_index_audio[clean_id]:
                audio_found.append({"missing_id": file_id, "found_path": str(found_path), "matched_as": clean_id})
                dest = DIR_RECOVERED / found_path.name
                if not dest.exists():
                    shutil.copy2(found_path, dest)
                    audio_copied += 1
            found = True
    if not found:
        audio_not_found.append(file_id)

print("=== MISSING AUDIO FILES ===")
print(f"Found: {len(audio_found)} / {len(missing_audio_ids)}")
print(f"Not found: {len(audio_not_found)} / {len(missing_audio_ids)}")

if audio_not_found:
    print("\nTruly missing audio files (not found anywhere):")
    for f in audio_not_found:
        print(f"  - {f}")

print(f"Copied {audio_copied} audio files to {DIR_RECOVERED}")

df_audio_found = pd.DataFrame(audio_found)

if not df_audio_found.empty:
    print("\nFound audio files:")
    display(df_audio_found)

=== MISSING AUDIO FILES ===
Found: 549 / 115
Not found: 3 / 115

Truly missing audio files (not found anywhere):
  - AU20
  - transcribed_social_research_interviews_73112
  - transcribed_social_research_interviews_73113
Copied 112 audio files to /Users/dmatekenya/Library/CloudStorage/GoogleDrive-dmatekenya@gmail.com/My Drive/Chichewa-ASR2/data/chichewa-speech-data/recovered_missing_files

Found audio files:


,missing_id,found_path,matched_as
0,220422-162004_nya_e46,/Users/dmatekenya/Library/CloudStorage/GoogleD...,NaN
1,220422-162004_nya_e46,/Users/dmatekenya/Library/CloudStorage/GoogleD...,NaN
2,220422-162004_nya_e46,/Users/dmatekenya/Library/CloudStorage/GoogleD...,NaN
3,220422-162004_nya_e46,/Users/dmatekenya/Library/CloudStorage/GoogleD...,NaN
4,220422-180143_nya_e46 (1),/Users/dmatekenya/Library/CloudStorage/GoogleD...,220422-180143_nya_e46
...,...,...,...
544,transcribed_social_research_interviews_775,/Users/dmatekenya/Library/CloudStorage/GoogleD...,NaN
545,transcribed_social_research_interviews_775,/Users/dmatekenya/Library/CloudStorage/GoogleD...,NaN
546,transcribed_social_research_interviews_775,/Users/dmatekenya/Library/CloudStorage/GoogleD...,NaN
547,transcribed_social_research_interviews_775,/Users/dmatekenya/Library/CloudStorage/GoogleD...,NaN


In [21]:
# Step 4: Search for missing transcript files (.txt) and copy found ones
missing_transc_ids = df_missing['no_transcript'].dropna().tolist()

transc_found = []
transc_not_found = []
transc_copied = 0

for file_id in missing_transc_ids:
    file_id = str(file_id).strip()
    found = False
    if file_id in file_index_txt:
        for found_path in file_index_txt[file_id]:
            transc_found.append({"missing_id": file_id, "found_path": str(found_path)})
            dest = DIR_RECOVERED / found_path.name
            if not dest.exists():
                shutil.copy2(found_path, dest)
                transc_copied += 1
        found = True
    else:
        # Try without " (1)" suffix
        clean_id = re.sub(r'\s*\(\d+\)$', '', file_id)
        if clean_id != file_id and clean_id in file_index_txt:
            for found_path in file_index_txt[clean_id]:
                transc_found.append({"missing_id": file_id, "found_path": str(found_path), "matched_as": clean_id})
                dest = DIR_RECOVERED / found_path.name
                if not dest.exists():
                    shutil.copy2(found_path, dest)
                    transc_copied += 1
            found = True
    if not found:
        transc_not_found.append(file_id)

print("=== MISSING TRANSCRIPT FILES (.txt) ===")
print(f"Found: {len(transc_found)} / {len(missing_transc_ids)}")
print(f"Not found: {len(transc_not_found)} / {len(missing_transc_ids)}")

if transc_not_found:
    print("\nTruly missing transcript files (not found anywhere):")
    for f in transc_not_found:
        print(f"  - {f}")

print(f"Copied {transc_copied} transcript files to {DIR_RECOVERED}")

df_transc_found = pd.DataFrame(transc_found)

if not df_transc_found.empty:
    print("\nFound transcript files:")
    display(df_transc_found)

=== MISSING TRANSCRIPT FILES (.txt) ===
Found: 237 / 113
Not found: 0 / 113
Copied 113 transcript files to /Users/dmatekenya/Library/CloudStorage/GoogleDrive-dmatekenya@gmail.com/My Drive/Chichewa-ASR2/data/chichewa-speech-data/recovered_missing_files

Found transcript files:


,missing_id,found_path,matched_as
0,220422-123447_nya_e46,/Users/dmatekenya/Library/CloudStorage/GoogleD...,NaN
1,220422-123447_nya_e46,/Users/dmatekenya/Library/CloudStorage/GoogleD...,NaN
2,220422-151734_nya_e46,/Users/dmatekenya/Library/CloudStorage/GoogleD...,NaN
3,220422-151734_nya_e46,/Users/dmatekenya/Library/CloudStorage/GoogleD...,NaN
4,220423-103230_nya_e46,/Users/dmatekenya/Library/CloudStorage/GoogleD...,NaN
...,...,...,...
232,transcribed_social_research_interviews_776,/Users/dmatekenya/Library/CloudStorage/GoogleD...,NaN
233,transcribed_social_research_interviews_776,/Users/dmatekenya/Library/CloudStorage/GoogleD...,NaN
234,transcribed_social_research_interviews_789,/Users/dmatekenya/Library/CloudStorage/GoogleD...,NaN
235,transcribed_social_research_interviews_789,/Users/dmatekenya/Library/CloudStorage/GoogleD...,NaN


In [22]:
# Step 5: Summary report and export
summary_data = []

for row in audio_found:
    summary_data.append({"file_id": row["missing_id"], "type": "audio (.wav)", 
                         "status": "FOUND", "path": row["found_path"]})
for f in audio_not_found:
    summary_data.append({"file_id": f, "type": "audio (.wav)", 
                         "status": "NOT FOUND", "path": None})
for row in transc_found:
    summary_data.append({"file_id": row["missing_id"], "type": "transcript (.txt)", 
                         "status": "FOUND", "path": row["found_path"]})
for f in transc_not_found:
    summary_data.append({"file_id": f, "type": "transcript (.txt)", 
                         "status": "NOT FOUND", "path": None})

df_report = pd.DataFrame(summary_data)

print("=== OVERALL SUMMARY ===")
print(df_report.groupby(["type", "status"]).size().unstack(fill_value=0))

# Save report
report_path = DIR_DATA.joinpath("missing_files_search_report.csv")
df_report.to_csv(report_path, index=False)
print(f"\nReport saved to: {report_path}")

print(f"\nAll recovered files copied to: {DIR_RECOVERED}")
print(f"Total files in recovery folder: {len(list(DIR_RECOVERED.iterdir()))}")

=== OVERALL SUMMARY ===
status             FOUND  NOT FOUND
type                               
audio (.wav)         549          3
transcript (.txt)    237          0

Report saved to: /Users/dmatekenya/git-repos/chichewa-asr/data/missing_files_search_report.csv

All recovered files copied to: /Users/dmatekenya/Library/CloudStorage/GoogleDrive-dmatekenya@gmail.com/My Drive/Chichewa-ASR2/data/chichewa-speech-data/recovered_missing_files
Total files in recovery folder: 225


## Build Clean Deduplicated metadata.csv
Recursively scan all folders under `DIR_DATA_GOOGLE_DRIVE`, deduplicate audio files by base name, match transcripts, compute duration, and flag files ≤30 seconds.

In [23]:
# Step A: Recursively scan and build file indices
from collections import defaultdict

AUDIO_EXT = {'.wav', '.mp3', '.m4a', '.aac', '.ogg', '.flac', '.mp4', '.opus'}
# Folders to skip (generated outputs, not source data)
SKIP_FOLDERS = {'recovered_missing_files', 'sample_5hr'}

audio_index = defaultdict(list)   # base_name -> [full_paths]
txt_index = defaultdict(list)     # base_name -> [full_paths]

scan_root = DIR_DATA_GOOGLE_DRIVE
print(f"Scanning: {scan_root}")

for fpath in scan_root.rglob("*"):
    if fpath.is_file():
        # Skip generated/output folders
        if any(skip in fpath.parts for skip in SKIP_FOLDERS):
            continue
        base_name = fpath.stem
        if fpath.suffix.lower() in AUDIO_EXT:
            audio_index[base_name].append(fpath)
        elif fpath.suffix.lower() == ".txt":
            txt_index[base_name].append(fpath)

print(f"Found {len(audio_index)} unique audio base names")
print(f"Found {len(txt_index)} unique .txt base names")

Scanning: /Users/dmatekenya/Library/CloudStorage/GoogleDrive-dmatekenya@gmail.com/My Drive/Chichewa-ASR2/data/chichewa-speech-data
Found 19924 unique audio base names
Found 4737 unique .txt base names


In [24]:
# Step B: Deduplicate — pick one canonical path per base name
# Priority: prefer files in chich-audio-transcript-files-v1, then first found
PRIORITY_FOLDER = "chich-audio-transcript-files-v1"

def pick_canonical(paths, priority_folder=PRIORITY_FOLDER):
    """Pick the best path: prefer files in the priority folder, else first."""
    for p in paths:
        if priority_folder in p.parts:
            return p
    return paths[0]

canonical_audio = {}  # base_name -> single Path
duplicates_log = []

for base_name, paths in audio_index.items():
    canonical_audio[base_name] = pick_canonical(paths)
    if len(paths) > 1:
        duplicates_log.append({
            "base_name": base_name,
            "num_copies": len(paths),
            "locations": [str(p.parent) for p in paths]
        })

print(f"Unique audio files (deduplicated): {len(canonical_audio)}")
print(f"Files with duplicates: {len(duplicates_log)}")
if duplicates_log:
    df_dupes = pd.DataFrame(duplicates_log)
    print(f"\nTop 10 duplicated files:")
    display(df_dupes.head(10))

Unique audio files (deduplicated): 19924
Files with duplicates: 10863

Top 10 duplicated files:


,base_name,num_copies,locations
0,dunstan_self_recorded_256,4,[/Users/dmatekenya/Library/CloudStorage/Google...
1,priscilla_self_records548,4,[/Users/dmatekenya/Library/CloudStorage/Google...
2,asante_records_99,4,[/Users/dmatekenya/Library/CloudStorage/Google...
3,dunstan_self_recorded_242,4,[/Users/dmatekenya/Library/CloudStorage/Google...
4,priscilla_self_records206,4,[/Users/dmatekenya/Library/CloudStorage/Google...
5,priscilla_self_records560,4,[/Users/dmatekenya/Library/CloudStorage/Google...
6,andrew_records_187,4,[/Users/dmatekenya/Library/CloudStorage/Google...
7,andrew_records_193,4,[/Users/dmatekenya/Library/CloudStorage/Google...
8,priscilla_self_records574,4,[/Users/dmatekenya/Library/CloudStorage/Google...
9,220423-154023_nya_e46,4,[/Users/dmatekenya/Library/CloudStorage/Google...


In [26]:
# Step C: Build metadata — match transcripts, compute duration, flag <=30s
metadata_rows = []
errors = []
total = len(canonical_audio)

for i, (base_name, audio_path) in enumerate(canonical_audio.items()):
    if (i + 1) % 200 == 0:
        print(f"Processing {i+1}/{total}...")

    # Audio filename
    audio_filename = audio_path.name

    # Match transcript
    if base_name in txt_index:
        txt_path = pick_canonical(txt_index[base_name])
        transcript_filename = txt_path.name
    else:
        transcript_filename = np.nan

    # Get duration
    try:
        audio = AudioSegment.from_file(audio_path)
        duration_seconds = round(len(audio) / 1000.0, 2)
    except Exception as e:
        errors.append({"audio_filename": audio_filename, "error": str(e)})
        duration_seconds = np.nan

    # Flag <=30 seconds
    if pd.notna(duration_seconds):
        thirty_seconds = "Yes" if duration_seconds <= 30 else "No"
    else:
        thirty_seconds = np.nan

    metadata_rows.append({
        "audio_filename": audio_filename,
        "transcript_filename": transcript_filename,
        "duration_seconds": duration_seconds,
        "thirty_seconds": thirty_seconds,
        "audio_filepath": str(audio_path),
    })

df_metadata_clean = pd.DataFrame(metadata_rows)
print(f"\nDone! Processed {len(df_metadata_clean)} unique audio files.")
if errors:
    print(f"Errors reading {len(errors)} files:")
    for err in errors[:10]:
        print(f"  - {err['audio_filename']}: {err['error']}")

Processing 200/19924...
Processing 400/19924...
Processing 600/19924...
Processing 800/19924...
Processing 1000/19924...
Processing 1200/19924...
Processing 1400/19924...
Processing 1600/19924...
Processing 1800/19924...
Processing 2000/19924...
Processing 2200/19924...
Processing 2400/19924...
Processing 2600/19924...
Processing 2800/19924...
Processing 3000/19924...
Processing 3200/19924...
Processing 3400/19924...
Processing 3600/19924...
Processing 3800/19924...
Processing 4000/19924...
Processing 4200/19924...
Processing 4400/19924...
Processing 4600/19924...
Processing 4800/19924...
Processing 5000/19924...
Processing 5200/19924...
Processing 5400/19924...
Processing 5600/19924...
Processing 5800/19924...
Processing 6000/19924...
Processing 6200/19924...
Processing 6400/19924...
Processing 6600/19924...
Processing 6800/19924...
Processing 7000/19924...
Processing 7200/19924...
Processing 7400/19924...
Processing 7600/19924...
Processing 7800/19924...
Processing 8000/19924...
Proc

In [27]:
# Step D: Summary stats and save
print("=== METADATA SUMMARY ===")
print(f"Total unique audio files: {len(df_metadata_clean)}")
print(f"With transcript: {df_metadata_clean['transcript_filename'].notna().sum()}")
print(f"Without transcript: {df_metadata_clean['transcript_filename'].isna().sum()}")
print(f"Audio ≤30 seconds: {(df_metadata_clean['thirty_seconds'] == 'Yes').sum()}")
print(f"Audio >30 seconds: {(df_metadata_clean['thirty_seconds'] == 'No').sum()}")

valid_durations = df_metadata_clean['duration_seconds'].dropna()
if not valid_durations.empty:
    print(f"\nTotal duration: {valid_durations.sum()/3600:.2f} hours")
    print(f"Mean duration: {valid_durations.mean():.1f}s | Median: {valid_durations.median():.1f}s")
    print(f"Min: {valid_durations.min():.1f}s | Max: {valid_durations.max():.1f}s")

# Save — drop the full filepath column from the saved CSV (keep for inspection)
metadata_out_path = DIR_DATA_GOOGLE_DRIVE / "metadata.csv"
df_metadata_clean.drop(columns=["audio_filepath"]).to_csv(metadata_out_path, index=False)
print(f"\nSaved to: {metadata_out_path}")

# Also save a copy to the repo data folder
repo_metadata_path = DIR_DATA / "metadata.csv"
df_metadata_clean.drop(columns=["audio_filepath"]).to_csv(repo_metadata_path, index=False)
print(f"Repo copy saved to: {repo_metadata_path}")

df_metadata_clean.head(10)

=== METADATA SUMMARY ===
Total unique audio files: 19924
With transcript: 4691
Without transcript: 15233
Audio ≤30 seconds: 16171
Audio >30 seconds: 313

Total duration: 121.06 hours
Mean duration: 26.4s | Median: 15.0s
Min: 0.1s | Max: 6681.8s

Saved to: /Users/dmatekenya/Library/CloudStorage/GoogleDrive-dmatekenya@gmail.com/My Drive/Chichewa-ASR2/data/chichewa-speech-data/metadata.csv
Repo copy saved to: /Users/dmatekenya/git-repos/chichewa-asr/data/metadata.csv


,audio_filename,transcript_filename,duration_seconds,thirty_seconds,audio_filepath
0,dunstan_self_recorded_256.wav,dunstan_self_recorded_256.txt,3.39,Yes,/Users/dmatekenya/Library/CloudStorage/GoogleD...
1,priscilla_self_records548.wav,priscilla_self_records548.txt,6.41,Yes,/Users/dmatekenya/Library/CloudStorage/GoogleD...
2,asante_records_99.wav,asante_records_99.txt,4.76,Yes,/Users/dmatekenya/Library/CloudStorage/GoogleD...
3,dunstan_self_recorded_242.wav,dunstan_self_recorded_242.txt,6.10,Yes,/Users/dmatekenya/Library/CloudStorage/GoogleD...
4,priscilla_self_records206.wav,priscilla_self_records206.txt,4.16,Yes,/Users/dmatekenya/Library/CloudStorage/GoogleD...
5,priscilla_self_records560.wav,priscilla_self_records560.txt,8.13,Yes,/Users/dmatekenya/Library/CloudStorage/GoogleD...
6,andrew_records_187.wav,andrew_records_187.txt,20.71,Yes,/Users/dmatekenya/Library/CloudStorage/GoogleD...
7,andrew_records_193.wav,andrew_records_193.txt,22.41,Yes,/Users/dmatekenya/Library/CloudStorage/GoogleD...
8,priscilla_self_records574.wav,priscilla_self_records574.txt,3.18,Yes,/Users/dmatekenya/Library/CloudStorage/GoogleD...
9,220423-154023_nya_e46.wav,220423-154023_nya_e46.txt,4.81,Yes,/Users/dmatekenya/Library/CloudStorage/GoogleD...


In [36]:
# Filter audio files with duration <= 45 seconds
df_45sec = df_metadata_clean[df_metadata_clean['duration_seconds'] <= 45]
df_over_45sec = df_metadata_clean[df_metadata_clean['duration_seconds'] > 45]

total_45sec_hours = df_45sec['duration_seconds'].sum() / 3600
total_over_45sec_hours = df_over_45sec['duration_seconds'].sum() / 3600

print("=== DURATION BREAKDOWN BY 45-SECOND THRESHOLD ===")
print(f"\nAudio files <= 45 seconds:")
print(f"  Count: {len(df_45sec):,}")
print(f"  Total duration: {total_45sec_hours:.2f} hours ({df_45sec['duration_seconds'].sum():,.0f} seconds)")
print(f"  Mean duration: {df_45sec['duration_seconds'].mean():.1f}s | Median: {df_45sec['duration_seconds'].median():.1f}s")

print(f"\nAudio files > 45 seconds:")
print(f"  Count: {len(df_over_45sec):,}")
print(f"  Total duration: {total_over_45sec_hours:.2f} hours ({df_over_45sec['duration_seconds'].sum():,.0f} seconds)")
print(f"  Mean duration: {df_over_45sec['duration_seconds'].mean():.1f}s | Median: {df_over_45sec['duration_seconds'].median():.1f}s")

total_hours = df_metadata_clean['duration_seconds'].sum() / 3600
print(f"\nTotal across all files: {total_hours:.2f} hours")
print(f"  <= 45s share: {total_45sec_hours/total_hours*100:.1f}%")
print(f"  >  45s share: {total_over_45sec_hours/total_hours*100:.1f}%")

=== DURATION BREAKDOWN BY 45-SECOND THRESHOLD ===

Audio files <= 45 seconds:
  Count: 16,319
  Total duration: 53.65 hours (193,130 seconds)
  Mean duration: 11.8s | Median: 15.0s

Audio files > 45 seconds:
  Count: 165
  Total duration: 67.41 hours (242,686 seconds)
  Mean duration: 1470.8s | Median: 263.6s

Total across all files: 121.06 hours
  <= 45s share: 44.3%
  >  45s share: 55.7%


In [41]:
# Create target folder
consolidated_dir = DIR_DATA_GOOGLE_DRIVE / "consolidated_with_transcripts_60s"
consolidated_dir.mkdir(parents=True, exist_ok=True)

# Keep only files <=60s and with transcript + audio path available
df_60s_with_transcript = df_metadata_clean[
    (df_metadata_clean["duration_seconds"].notna()) &
    (df_metadata_clean["duration_seconds"] <= 60) &
    (df_metadata_clean["transcript_filename"].notna()) &
    (df_metadata_clean["audio_filepath"].notna())
].copy()

audio_copied = 0
txt_copied = 0
missing_audio = []
missing_txt = []

for _, row in df_60s_with_transcript.iterrows():
    audio_path = Path(row["audio_filepath"])
    base_name = audio_path.stem

    # Resolve transcript path using txt index first, then fallback near audio file
    transcript_path = None
    txt_candidates = txt_index.get(base_name, [])
    if txt_candidates:
        transcript_path = pick_canonical(txt_candidates)
    else:
        fallback_txt = audio_path.with_suffix(".txt")
        if fallback_txt.exists():
            transcript_path = fallback_txt

    if not audio_path.exists():
        missing_audio.append(str(audio_path))
        continue

    if transcript_path is None or not Path(transcript_path).exists():
        missing_txt.append(base_name)
        continue

    dest_audio = consolidated_dir / audio_path.name
    dest_txt = consolidated_dir / Path(transcript_path).name

    if not dest_audio.exists():
        shutil.copy2(audio_path, dest_audio)
        audio_copied += 1

    if not dest_txt.exists():
        shutil.copy2(transcript_path, dest_txt)
        txt_copied += 1

print("=== CONSOLIDATION COMPLETE (<=60s) ===")
print(f"Target folder: {consolidated_dir}")
print(f"Eligible rows: {len(df_60s_with_transcript)}")
print(f"Audio copied: {audio_copied}")
print(f"Transcripts copied: {txt_copied}")
print(f"Missing audio paths: {len(missing_audio)}")
print(f"Missing transcripts: {len(missing_txt)}")

=== CONSOLIDATION COMPLETE (<=60s) ===
Target folder: /Users/dmatekenya/Library/CloudStorage/GoogleDrive-dmatekenya@gmail.com/My Drive/Chichewa-ASR2/data/chichewa-speech-data/consolidated_with_transcripts_60s
Eligible rows: 4618
Audio copied: 4618
Transcripts copied: 4618
Missing audio paths: 0
Missing transcripts: 0


In [42]:
# Step 1: Calculate total duration of files in consolidated_with_transcripts_45s
consolidated_audio_files = [f for f in consolidated_dir.iterdir() if f.suffix.lower() in {'.wav', '.mp3', '.m4a', '.aac', '.ogg', '.flac', '.mp4', '.opus'}]
total_duration_consolidated = 0
audio_durations = {}

for audio_file in consolidated_audio_files:
    try:
        audio = AudioSegment.from_file(audio_file)
        duration_seconds = len(audio) / 1000.0
        total_duration_consolidated += duration_seconds
        audio_durations[audio_file.stem] = duration_seconds
    except Exception as e:
        print(f"Error reading {audio_file.name}: {e}")

print(f"Total duration in consolidated folder: {total_duration_consolidated/3600:.2f} hours ({total_duration_consolidated:,.0f} seconds)")
print(f"Total files: {len(consolidated_audio_files)}")

# Step 2: Create metadata with word/character counts from transcripts
consolidated_metadata = []

for audio_file in consolidated_audio_files:
    base_name = audio_file.stem
    txt_file = consolidated_dir / f"{base_name}.txt"
    
    duration_seconds = audio_durations.get(base_name, np.nan)
    
    if txt_file.exists():
        try:
            with open(txt_file, 'r', encoding='utf-8', errors='replace') as f:
                transcript_text = f.read().strip()
            word_count = len(transcript_text.split())
            char_count = len(transcript_text)
            words_per_second = word_count / duration_seconds if pd.notna(duration_seconds) else np.nan
        except Exception as e:
            transcript_text = None
            word_count = np.nan
            char_count = np.nan
            words_per_second = np.nan
            print(f"Error reading transcript {txt_file.name}: {e}")
    else:
        transcript_text = None
        word_count = np.nan
        char_count = np.nan
        words_per_second = np.nan
    
    consolidated_metadata.append({
        'audio_filename': audio_file.name,
        'transcript_filename': txt_file.name if txt_file.exists() else np.nan,
        'duration_seconds': duration_seconds,
        'word_count': word_count,
        'char_count': char_count,
        'words_per_second': words_per_second
    })

df_consolidated_metadata = pd.DataFrame(consolidated_metadata)

# Save metadata
consolidated_metadata_path = consolidated_dir / "metadata.csv"
df_consolidated_metadata.to_csv(consolidated_metadata_path, index=False)
print(f"\nMetadata saved to: {consolidated_metadata_path}")

# Step 3: Summary and identify mismatches
print("\n=== CONSOLIDATED METADATA SUMMARY ===")
print(f"Total files: {len(df_consolidated_metadata)}")
print(f"\nWord count statistics:")
print(f"  Mean: {df_consolidated_metadata['word_count'].mean():.1f}")
print(f"  Median: {df_consolidated_metadata['word_count'].median():.1f}")
print(f"  Min: {df_consolidated_metadata['word_count'].min():.0f}")
print(f"  Max: {df_consolidated_metadata['word_count'].max():.0f}")

print(f"\nWords per second statistics:")
print(f"  Mean: {df_consolidated_metadata['words_per_second'].mean():.2f}")
print(f"  Median: {df_consolidated_metadata['words_per_second'].median():.2f}")
print(f"  Min: {df_consolidated_metadata['words_per_second'].min():.2f}")
print(f"  Max: {df_consolidated_metadata['words_per_second'].max():.2f}")

# Identify potential mismatches (e.g., very low or very high words per second)
wps_mean = df_consolidated_metadata['words_per_second'].mean()
wps_std = df_consolidated_metadata['words_per_second'].std()
potential_mismatches = df_consolidated_metadata[
    (df_consolidated_metadata['words_per_second'] < wps_mean - 2*wps_std) |
    (df_consolidated_metadata['words_per_second'] > wps_mean + 2*wps_std)
]

print(f"\n=== POTENTIAL MISMATCHES ===")
print(f"Files with unusual words-per-second ratio (outside ±2 std): {len(potential_mismatches)}")
if not potential_mismatches.empty:
    print("\nTop 10 mismatches:")
    display(potential_mismatches.nlargest(10, 'words_per_second')[['audio_filename', 'duration_seconds', 'word_count', 'words_per_second']])


Total duration in consolidated folder: 15.57 hours (56,035 seconds)
Total files: 4618

Metadata saved to: /Users/dmatekenya/Library/CloudStorage/GoogleDrive-dmatekenya@gmail.com/My Drive/Chichewa-ASR2/data/chichewa-speech-data/consolidated_with_transcripts_60s/metadata.csv

=== CONSOLIDATED METADATA SUMMARY ===
Total files: 4618

Word count statistics:
  Mean: 20.9
  Median: 19.0
  Min: 0
  Max: 3154

Words per second statistics:
  Mean: 2.01
  Median: 1.58
  Min: 0.00
  Max: 924.11

=== POTENTIAL MISMATCHES ===
Files with unusual words-per-second ratio (outside ±2 std): 3

Top 10 mismatches:


,audio_filename,duration_seconds,word_count,words_per_second
2942,priscilla_self_records108.wav,3.413,3154,924.113683
3090,asante_records_267.wav,2.694,2247,834.075724
4332,dunstan_self_recorded_266.wav,7.872,1193,151.549797


In [50]:
# Step 1: Get list of audio files already in consolidated_dir
consolidated_audio_basenames = {Path(f).stem for f in consolidated_dir.iterdir() if f.suffix.lower() in {'.wav', '.mp3', '.m4a', '.aac', '.ogg', '.flac', '.mp4', '.opus'}}

print(f"Files already in consolidated_dir: {len(consolidated_audio_basenames)}")

# Step 2: Scan DIR_GOOGLE_SUBMISSIONS for audio and transcript files
submission_audio_files = []
submission_txt_files = []

for fpath in DIR_GOOGLE_SUBMISSIONS.rglob("*"):
    if fpath.is_file():
        if fpath.suffix.lower() in {'.wav', '.mp3', '.m4a', '.aac', '.ogg', '.flac', '.mp4', '.opus'}:
            submission_audio_files.append(fpath)
        elif fpath.suffix.lower() == ".txt":
            submission_txt_files.append(fpath)

print(f"\nFound in DIR_GOOGLE_SUBMISSIONS:")
print(f"  Audio files: {len(submission_audio_files)}")
print(f"  Transcript files: {len(submission_txt_files)}")

# Step 3: Copy only audio files that don't already exist in consolidated_dir
audio_copied_google = 0
txt_copied_google = 0

for audio_file in submission_audio_files:
    base_name = audio_file.stem
    
    # Skip if already exists
    if base_name in consolidated_audio_basenames:
        print(f"Skipping {audio_file.name} (already exists)")
        continue
    
    dest_audio = consolidated_dir / audio_file.name
    shutil.copy2(audio_file, dest_audio)
    audio_copied_google += 1
    
    # Look for matching transcript
    for txt_file in submission_txt_files:
        if txt_file.stem == base_name:
            dest_txt = consolidated_dir / txt_file.name
            if not dest_txt.exists():
                shutil.copy2(txt_file, dest_txt)
                txt_copied_google += 1
            break

print(f"\n=== COPY FROM DIR_GOOGLE_SUBMISSIONS ===")
print(f"Audio files copied: {audio_copied_google}")
print(f"Transcript files copied: {txt_copied_google}")

# Step 4: Recalculate summary for consolidated_dir
consolidated_audio_files = [f for f in consolidated_dir.iterdir() if f.suffix.lower() in {'.wav', '.mp3', '.m4a', '.aac', '.ogg', '.flac', '.mp4', '.opus'}]
total_duration_consolidated_updated = 0

for audio_file in consolidated_audio_files:
    try:
        audio = AudioSegment.from_file(audio_file)
        total_duration_consolidated_updated += len(audio) / 1000.0
    except Exception as e:
        print(f"Error reading {audio_file.name}: {e}")

print(f"\n=== CONSOLIDATED DIRECTORY SUMMARY (UPDATED) ===")
print(f"Total audio files: {len(consolidated_audio_files)}")
print(f"Total duration: {total_duration_consolidated_updated/3600:.2f} hours ({total_duration_consolidated_updated:,.0f} seconds)")

Files already in consolidated_dir: 4618

Found in DIR_GOOGLE_SUBMISSIONS:
  Audio files: 342
  Transcript files: 343
Skipping AU23.wav (already exists)
Skipping WAU106.wav (already exists)
Skipping WAU112.wav (already exists)
Skipping AU37.wav (already exists)
Skipping WAU6.wav (already exists)
Skipping AU111.wav (already exists)
Skipping AU105.wav (already exists)
Skipping AU139.wav (already exists)
Skipping WAU58.wav (already exists)
Skipping WAU64.wav (already exists)
Skipping WAU65.wav (already exists)
Skipping WAU71.wav (already exists)
Skipping WAU59.wav (already exists)
Skipping AU138.wav (already exists)
Skipping AU104.wav (already exists)
Skipping AU110.wav (already exists)
Skipping WAU7.wav (already exists)
Skipping WAU113.wav (already exists)
Skipping AU36.wav (already exists)
Skipping AU22.wav (already exists)
Skipping WAU107.wav (already exists)
Skipping AU34.wav (already exists)
Skipping WAU111.wav (already exists)
Skipping WAU105.wav (already exists)
Skipping WAU139.wav 

In [54]:
# Create a new 60s folder with only files <= 60 seconds
consolidated_60s_dir = DIR_DATA_GOOGLE_DRIVE / "consolidated_with_transcripts_60s_filtered"
consolidated_60s_dir.mkdir(parents=True, exist_ok=True)

# Filter dataframe for files <= 60 seconds
df_60s_filtered = df_consolidated_updated[df_consolidated_updated['duration_seconds'] <= 60].copy()

audio_copied_60s = 0
txt_copied_60s = 0

for _, row in df_60s_filtered.iterrows():
    audio_filename = row['audio_filename']
    transcript_filename = row['transcript_filename']
    
    src_audio = consolidated_dir / audio_filename
    dest_audio = consolidated_60s_dir / audio_filename
    
    if src_audio.exists() and not dest_audio.exists():
        shutil.copy2(src_audio, dest_audio)
        audio_copied_60s += 1
    
    if pd.notna(transcript_filename):
        src_txt = consolidated_dir / transcript_filename
        dest_txt = consolidated_60s_dir / transcript_filename
        
        if src_txt.exists() and not dest_txt.exists():
            shutil.copy2(src_txt, dest_txt)
            txt_copied_60s += 1

print("=== NEW 60-SECOND FILTERED FOLDER ===")
print(f"Target folder: {consolidated_60s_dir}")
print(f"Files <= 60 seconds: {len(df_60s_filtered)}")
print(f"Audio files copied: {audio_copied_60s}")
print(f"Transcript files copied: {txt_copied_60s}")

total_duration_60s = df_60s_filtered['duration_seconds'].sum()
print(f"Total duration: {total_duration_60s/3600:.2f} hours ({total_duration_60s:,.0f} seconds)")


=== NEW 60-SECOND FILTERED FOLDER ===
Target folder: /Users/dmatekenya/Library/CloudStorage/GoogleDrive-dmatekenya@gmail.com/My Drive/Chichewa-ASR2/data/chichewa-speech-data/consolidated_with_transcripts_60s_filtered
Files <= 60 seconds: 4620
Audio files copied: 4620
Transcript files copied: 4620
Total duration: 15.58 hours (56,103 seconds)


In [56]:
str(consolidated_60s_dir)


'/Users/dmatekenya/Library/CloudStorage/GoogleDrive-dmatekenya@gmail.com/My Drive/Chichewa-ASR2/data/chichewa-speech-data/consolidated_with_transcripts_60s_filtered'

## Build Rich Metadata for consolidated_with_transcripts_60s_filtered
Compute duration, transcript stats, speaker info, sex, num_speakers, and duration buckets.

In [82]:
# Build rich metadata for consolidated_60s_dir
target_dir = consolidated_60s_dir

AUDIO_EXT_SET = {'.wav', '.mp3', '.m4a', '.aac', '.ogg', '.flac', '.mp4', '.opus'}
audio_files_list = sorted([f for f in target_dir.iterdir() if f.suffix.lower() in AUDIO_EXT_SET])

print(f"Audio files in {target_dir.name}: {len(audio_files_list)}")

# --- Speaker grouping strategy ---
# Files were created by splitting longer audio into chunks. The split function
# appends the chunk index directly to the base filename string:
#   AU1 -> AU10, AU11, ..., AU19, AU110, AU111, ..., AU142
#   WAU2 -> WAU20, WAU21, ..., WAU29, WAU210, WAU211, ...
#   transcribed_social_research_interviews_5 -> ..._50, ..._51, ..., ..._5385
#   LongAudio1 -> LongAudio10, ..., LongAudio1121
# The FIRST DIGIT of the trailing number identifies the original speaker/recording.
# This gives a clean lower-bound on unique speakers.
social_research_prefix = "transcribed_social_research_interviews_"

# --- Helper: extract speaker name from filename ---
def get_speaker_name(filename):
    stem = Path(filename).stem.lower()
    
    # Pattern: 220423-191801_nya_e46 -> dunstan
    if re.match(r'^\d{6}-\d{6}_nya_e\d+', stem):
        return "dunstan"
    
    # WAU files: first digit of number = speaker group
    m = re.match(r'^wau(\d+)', stem)
    if m:
        return f"wau_speaker_{m.group(1)[0]}"
    
    # AU files (not aud_from, not audio_from): first digit of number = speaker group
    m = re.match(r'^au(\d+)', stem)
    if m and not stem.startswith("aud_from") and not stem.startswith("audio"):
        return f"au_speaker_{m.group(1)[0]}"
    
    # WhatsApp files: base number after "whatsapp" (before _1/_2 split suffixes)
    m = re.search(r'whatsapp_?(?:vid_?)?(\d+)', stem)
    if m:
        return f"whatsapp_speaker_{m.group(1)}"
    if "whatsapp" in stem:
        return "whatsapp_speaker_unknown"
    
    # aud_from_vid files
    m = re.match(r'^aud_from_vid(\d+)', stem)
    if m:
        return f"vid_speaker_{m.group(1)}"
    
    # Social research interviews: first digit of number = interview/speaker
    if stem.startswith(social_research_prefix):
        num_part = stem[len(social_research_prefix):]
        if num_part.isdigit():
            return f"interview_speaker_{num_part[0]}"
    
    # LongAudio: first digit of number = speaker
    m = re.match(r'^longaudio(\d+)', stem)
    if m:
        return f"long_speaker_{m.group(1)[0]}"
    
    # Named speakers: split by "_", first word is the name
    parts = stem.split("_")
    if len(parts) >= 2:
        candidate = parts[0].strip()
        if candidate and candidate.isalpha():
            return candidate
    
    return "unknown"

# --- Helper: get num_speakers ---
def get_num_speakers(filename, speaker_name):
    stem = Path(filename).stem.lower()
    
    # Social research / long interviews -> 2 speakers (interviewer + interviewee)
    if any(kw in stem for kw in ["long", "social", "research"]):
        return 2
    
    # Known named speakers or identified unique speakers -> 1
    if speaker_name != "unknown":
        return 1
    
    return 9

# --- Helper: get sex ---
SEX_MAP = {
    "asante": "Male",
    "dunstan": "Male",
    "andrew": "Male",
    "priscilla": "Female",
}

def get_sex(speaker_name):
    return SEX_MAP.get(speaker_name.lower(), "unknown")

# --- Helper: duration bucket ---
def get_duration_bucket(duration_sec):
    if pd.isna(duration_sec):
        return np.nan
    if duration_sec <= 5:
        return 1
    elif duration_sec <= 10:
        return 2
    elif duration_sec <= 15:
        return 3
    elif duration_sec <= 20:
        return 4
    elif duration_sec <= 30:
        return 5
    else:
        return 6

# --- Build metadata ---
rich_metadata = []
errors_rich = []

for i, audio_file in enumerate(audio_files_list):
    if (i + 1) % 200 == 0:
        print(f"Processing {i+1}/{len(audio_files_list)}...")
    
    audio_fname = audio_file.name
    base_name = audio_file.stem
    
    # Duration
    try:
        audio_seg = AudioSegment.from_file(audio_file)
        dur_sec = round(len(audio_seg) / 1000.0, 2)
    except Exception as e:
        errors_rich.append({"file": audio_fname, "error": str(e)})
        dur_sec = np.nan
    
    # Transcript stats
    txt_path = target_dir / f"{base_name}.txt"
    if txt_path.exists():
        try:
            with open(txt_path, 'r', encoding='utf-8', errors='replace') as f:
                text = f.read().strip()
            t_word_count = len(text.split()) if text else 0
            t_char_count = len(text) if text else 0
            transcript_fname = txt_path.name
        except Exception:
            t_word_count = np.nan
            t_char_count = np.nan
            transcript_fname = np.nan
    else:
        t_word_count = np.nan
        t_char_count = np.nan
        transcript_fname = np.nan
    
    # Derived stats
    if pd.notna(dur_sec) and dur_sec > 0 and pd.notna(t_word_count):
        wps = round(t_word_count / dur_sec, 2)
        cps = round(t_char_count / dur_sec, 2)
    else:
        wps = np.nan
        cps = np.nan
    
    # Speaker info
    speaker = get_speaker_name(audio_fname)
    num_spk = get_num_speakers(audio_fname, speaker)
    sex = get_sex(speaker)
    
    # Duration bucket
    dur_bucket = get_duration_bucket(dur_sec)
    
    rich_metadata.append({
        "audio_filename": audio_fname,
        "transcript_filename": transcript_fname,
        "duration_seconds": dur_sec,
        "duration_bucket": dur_bucket,
        "transcript_word_count": t_word_count,
        "transcript_char_count": t_char_count,
        "words_per_second": wps,
        "characters_per_second": cps,
        "speaker_name": speaker,
        "speaker_sex": sex,
        "num_speakers": num_spk,
    })

df_rich = pd.DataFrame(rich_metadata)
print(f"\nDone! {len(df_rich)} files processed.")
if errors_rich:
    print(f"Errors: {len(errors_rich)}")
    for e in errors_rich[:5]:
        print(f"  - {e['file']}: {e['error']}")

Audio files in consolidated_with_transcripts_60s_filtered: 4620
Processing 200/4620...
Processing 400/4620...
Processing 600/4620...
Processing 800/4620...
Processing 1000/4620...
Processing 1200/4620...
Processing 1400/4620...
Processing 1600/4620...
Processing 1800/4620...
Processing 2000/4620...
Processing 2200/4620...
Processing 2400/4620...
Processing 2600/4620...
Processing 2800/4620...
Processing 3000/4620...
Processing 3200/4620...
Processing 3400/4620...
Processing 3600/4620...
Processing 3800/4620...
Processing 4000/4620...
Processing 4200/4620...
Processing 4400/4620...
Processing 4600/4620...

Done! 4620 files processed.


In [96]:
# Summary and save rich metadata
print("=== RICH METADATA SUMMARY ===")
print(f"Total files: {len(df_rich)}")

print(f"\n--- Duration ---")
print(f"  Total: {df_rich['duration_seconds'].sum()/3600:.2f} hours")
print(f"  Mean: {df_rich['duration_seconds'].mean():.1f}s | Median: {df_rich['duration_seconds'].median():.1f}s")

print(f"\n--- Duration Buckets ---")
bucket_labels = {1: "1-5s", 2: "5-10s", 3: "10-15s", 4: "15-20s", 5: "20-30s", 6: ">30s"}
for bucket, count in df_rich['duration_bucket'].value_counts().sort_index().items():
    print(f"  Bucket {int(bucket)} ({bucket_labels.get(bucket, '?')}): {count}")

print(f"\n--- Transcript Stats ---")
has_transcript = df_rich['transcript_filename'].notna().sum()
print(f"  With transcript: {has_transcript}")
print(f"  Word count — Mean: {df_rich['transcript_word_count'].mean():.1f} | Median: {df_rich['transcript_word_count'].median():.1f}")
print(f"  Words/sec — Mean: {df_rich['words_per_second'].mean():.2f} | Median: {df_rich['words_per_second'].median():.2f}")

print(f"\n--- Speakers ---")
print(f"  Unique speaker IDs: {df_rich['speaker_name'].nunique()}")
print(f"\n  Top 20 speakers by file count:")
print(df_rich['speaker_name'].value_counts().head(20).to_string())

print(f"\n--- Speaker Sex ---")
print(df_rich['speaker_sex'].value_counts().to_string())

print(f"\n--- Num Speakers ---")
print(df_rich['num_speakers'].value_counts().sort_index().to_string())

# Save
rich_meta_path = target_dir / "rich_metadata.csv"
df_rich.to_csv(rich_meta_path, index=False)
print(f"\nSaved to: {rich_meta_path}")

# Also save to repo data folder
repo_rich_path = DIR_DATA / "rich_metadata.csv"
df_rich.to_csv(repo_rich_path, index=False)
print(f"Repo copy: {repo_rich_path}")

df_rich.head(10)

=== RICH METADATA SUMMARY ===
Total files: 4620

--- Duration ---
  Total: 15.58 hours
  Mean: 12.1s | Median: 15.0s

--- Duration Buckets ---
  Bucket 1 (1-5s): 847
  Bucket 2 (5-10s): 874
  Bucket 3 (10-15s): 2307
  Bucket 4 (15-20s): 322
  Bucket 5 (20-30s): 216
  Bucket 6 (>30s): 54

--- Transcript Stats ---
  With transcript: 4620
  Word count — Mean: 20.9 | Median: 19.0
  Words/sec — Mean: 2.01 | Median: 1.58

--- Speakers ---
  Unique speaker IDs: 85

  Top 20 speakers by file count:
speaker_name
asante                 679
priscilla              609
dunstan                508
interview_speaker_5    384
andrew                 351
interview_speaker_6    331
interview_speaker_7    314
interview_speaker_4    260
interview_speaker_1    246
interview_speaker_2    229
interview_speaker_3    176
long_speaker_1         122
wau_speaker_1           79
au_speaker_1            54
wau_speaker_2           23
unknown                 19
wau_speaker_9           13
au_speaker_3            11
au_sp

,audio_filename,transcript_filename,duration_seconds,duration_bucket,transcript_word_count,transcript_char_count,words_per_second,characters_per_second,speaker_name,speaker_sex,num_speakers
0,220422-123447_nya_e46.wav,220422-123447_nya_e46.txt,5.97,2,5,36,0.84,6.03,dunstan,Male,1
1,220422-135445_nya_e46.wav,220422-135445_nya_e46.txt,6.96,2,7,51,1.01,7.33,dunstan,Male,1
2,220422-135511_nya_e46.wav,220422-135511_nya_e46.txt,9.98,2,11,77,1.10,7.72,dunstan,Male,1
3,220422-151734_nya_e46.wav,220422-151734_nya_e46.txt,12.37,3,17,124,1.37,10.02,dunstan,Male,1
4,220422-151915_nya_e46.wav,220422-151915_nya_e46.txt,11.73,3,17,103,1.45,8.78,dunstan,Male,1
5,220422-151957_nya_e46.wav,220422-151957_nya_e46.txt,9.96,2,12,77,1.20,7.73,dunstan,Male,1
6,220422-152145_nya_e46.wav,220422-152145_nya_e46.txt,5.85,2,6,47,1.03,8.03,dunstan,Male,1
7,220422-152224_nya_e46.wav,220422-152224_nya_e46.txt,9.48,2,10,68,1.05,7.17,dunstan,Male,1
8,220422-152257_nya_e46.wav,220422-152257_nya_e46.txt,11.12,3,11,86,0.99,7.73,dunstan,Male,1
9,220422-152711_nya_e46.wav,220422-152711_nya_e46.txt,6.62,2,7,43,1.06,6.50,dunstan,Male,1


In [111]:
def classify_row(row):
    # Drop extremely short audio
    if row["duration_seconds"] < 1:
        return "drop_short"

    # Out of scope for now
    if row["duration_seconds"] > 30:
        return "too_long"

    # High priority: likely annotator issue
    if (row["words_per_second"] < 0.5) or (
        (row["duration_seconds"] > 10) and (row["transcript_word_count"] < 5)
    ):
        return "high_priority"

    # Likely mismatch (audio/text misalignment)
    if row["words_per_second"] > 6:
        return "mismatch"

    return "ok"

In [112]:

df_rich["review_flag"] = df_rich.apply(classify_row, axis=1)

In [113]:
df_rich.review_flag.value_counts(normalize=True)

review_flag
ok               0.966450
high_priority    0.020779
too_long         0.011688
mismatch         0.000866
drop_short       0.000216
Name: proportion, dtype: float64

In [114]:
# Split df_rich by review_flag and copy files into two folders
source_dir = target_dir  # uses the existing source folder for df_rich files

dev_test_dir = DIR_DATA_GOOGLE_DRIVE / "dev_test_aims_cameroon"
evance_review_dir = DIR_DATA_GOOGLE_DRIVE / "evance_review"
dev_test_dir.mkdir(parents=True, exist_ok=True)
evance_review_dir.mkdir(parents=True, exist_ok=True)

df_rich_ok = df_rich[df_rich["review_flag"].eq("ok")].copy()
df_review = df_rich[~df_rich["review_flag"].eq("ok")].copy()  # kept for review and later analysis

def copy_audio_and_transcripts(df_subset, out_dir):
    audio_copied, txt_copied, missing_audio = 0, 0, 0
    for _, row in df_subset.iterrows():
        audio_name = row["audio_filename"]
        src_audio = source_dir / audio_name
        dst_audio = out_dir / audio_name

        if src_audio.exists():
            if not dst_audio.exists():
                shutil.copy2(src_audio, dst_audio)
                audio_copied += 1
        else:
            missing_audio += 1
            continue

        transcript_name = row.get("transcript_filename", np.nan)
        if pd.notna(transcript_name):
            src_txt = source_dir / transcript_name
            dst_txt = out_dir / transcript_name
            if src_txt.exists() and not dst_txt.exists():
                shutil.copy2(src_txt, dst_txt)
                txt_copied += 1

    return audio_copied, txt_copied, missing_audio

ok_audio_copied, ok_txt_copied, ok_missing_audio = copy_audio_and_transcripts(df_rich_ok, dev_test_dir)
rv_audio_copied, rv_txt_copied, rv_missing_audio = copy_audio_and_transcripts(df_review, evance_review_dir)

# Save updated metadata for dev_test_aims_cameroon
dev_metadata_path = dev_test_dir / "metadata.csv"
df_rich_ok.to_csv(dev_metadata_path, index=False)

print("=== SPLIT COMPLETE ===")
print(f"Source dir: {source_dir}")
print(f"OK set size: {len(df_rich_ok)} | Review set size: {len(df_review)}")
print(f"\nDev/Test folder: {dev_test_dir}")
print(f"  Audio copied: {ok_audio_copied}, Transcript copied: {ok_txt_copied}, Missing audio: {ok_missing_audio}")
print(f"  Metadata saved: {dev_metadata_path}")

print(f"\nReview folder: {evance_review_dir}")
print(f"  Audio copied: {rv_audio_copied}, Transcript copied: {rv_txt_copied}, Missing audio: {rv_missing_audio}")

=== SPLIT COMPLETE ===
Source dir: /Users/dmatekenya/Library/CloudStorage/GoogleDrive-dmatekenya@gmail.com/My Drive/Chichewa-ASR2/data/chichewa-speech-data/consolidated_with_transcripts_60s_filtered
OK set size: 4465 | Review set size: 155

Dev/Test folder: /Users/dmatekenya/Library/CloudStorage/GoogleDrive-dmatekenya@gmail.com/My Drive/Chichewa-ASR2/data/chichewa-speech-data/dev_test_aims_cameroon
  Audio copied: 4465, Transcript copied: 4465, Missing audio: 0
  Metadata saved: /Users/dmatekenya/Library/CloudStorage/GoogleDrive-dmatekenya@gmail.com/My Drive/Chichewa-ASR2/data/chichewa-speech-data/dev_test_aims_cameroon/metadata.csv

Review folder: /Users/dmatekenya/Library/CloudStorage/GoogleDrive-dmatekenya@gmail.com/My Drive/Chichewa-ASR2/data/chichewa-speech-data/evance_review
  Audio copied: 155, Transcript copied: 155, Missing audio: 0


In [5]:
dev_test_dir = DIR_DATA_GOOGLE_DRIVE / "dev_test_aims_cameroon"
dev_metadata_path = dev_test_dir / "metadata.csv"
df_rich_ok = pd.read_csv(dev_metadata_path)

In [7]:
total_pydub_seconds = df_duration_check["duration_pydub"].sum()
total_librosa_seconds = df_duration_check["duration_librosa"].sum()
total_metadata_seconds = df_duration_check["duration_metadata"].sum()

print("=== TOTAL DURATION BY SOURCE ===")
print(f"Pydub:    {total_pydub_seconds/3600:.2f} hours ({total_pydub_seconds:,.3f} seconds)")
print(f"Librosa:  {total_librosa_seconds/3600:.2f} hours ({total_librosa_seconds:,.3f} seconds)")
print(f"Metadata: {total_metadata_seconds/3600:.2f} hours ({total_metadata_seconds:,.2f} seconds)")

print("\n=== DIFFERENCES ===")
print(f"Pydub vs Librosa:  {abs(total_pydub_seconds - total_librosa_seconds):,.3f} seconds")
print(f"Pydub vs Metadata: {abs(total_pydub_seconds - total_metadata_seconds):,.3f} seconds")
print(f"Librosa vs Metadata: {abs(total_librosa_seconds - total_metadata_seconds):,.3f} seconds")


=== TOTAL DURATION BY SOURCE ===
Pydub:    14.69 hours (52,895.376 seconds)
Librosa:  14.69 hours (52,895.374 seconds)
Metadata: 14.69 hours (52,895.27 seconds)

=== DIFFERENCES ===
Pydub vs Librosa:  0.002 seconds
Pydub vs Metadata: 0.106 seconds
Librosa vs Metadata: 0.104 seconds


In [10]:
speaker_hours

NameError: name 'speaker_hours' is not defined

In [13]:
# Tabulate total duration (hours) by speaker in df_rich_ok
speaker_hours = (
    df_rich_ok
    .groupby("speaker_name", dropna=False)["duration_seconds"]
    .sum()
    .div(3600)
    .reset_index(name="duration_hours")
    .sort_values("duration_hours", ascending=False)
)

speaker_hours["duration_hours"] = speaker_hours["duration_hours"].round(2)

print(f"Total speakers: {speaker_hours.shape[0]}")
display(speaker_hours)

Total speakers: 77


,speaker_name,duration_hours
1,andrew,1.82
20,interview_speaker_5,1.58
21,interview_speaker_6,1.34
22,interview_speaker_7,1.30
2,asante,1.16
...,...,...
35,vid_speaker_15,0.00
66,whatsapp_speaker_14,0.00
47,vid_speaker_28,0.00
40,vid_speaker_21,0.00


In [17]:
dev_test_dir

PosixPath('/Users/dmatekenya/Library/CloudStorage/GoogleDrive-dmatekenya@gmail.com/My Drive/Chichewa-ASR2/data/chichewa-speech-data/dev_test_aims_cameroon')

In [8]:
dev_test_dir = DIR_DATA_GOOGLE_DRIVE / "dev_test_aims_cameroon"
female_speakers = DIR_DATA_GOOGLE_DRIVE / "dev_test_aims_cameroon /Female"
dev_metadata_path = dev_test_dir / "metadata.csv"
df_rich_ok = pd.read_csv(dev_metadata_path)

In [9]:
# Check durations of audio files in female_speakers folder
audio_exts = {'.wav', '.mp3', '.m4a', '.aac', '.ogg', '.flac', '.mp4', '.opus'}

female_dir = female_speakers if female_speakers.exists() else (dev_test_dir / "Female")
print(f"Using folder: {female_dir}")
print(f"Exists: {female_dir.exists()}")

female_audio_rows = []
female_audio_errors = []

if female_dir.exists():
    for fpath in female_dir.rglob("*"):
        if fpath.is_file() and fpath.suffix.lower() in audio_exts:
            try:
                audio = AudioSegment.from_file(fpath)
                female_audio_rows.append({
                    "audio_filename": fpath.name,
                    "duration_seconds": round(len(audio) / 1000.0, 2)
                })
            except Exception as e:
                female_audio_errors.append({"audio_filename": fpath.name, "error": str(e)})

df_female_duration = pd.DataFrame(female_audio_rows).sort_values(
    "duration_seconds", ascending=False
).reset_index(drop=True) if female_audio_rows else pd.DataFrame(
    columns=["audio_filename", "duration_seconds"]
)

print(f"\nAudio files found: {len(df_female_duration)}")
if not df_female_duration.empty:
    total_seconds = df_female_duration["duration_seconds"].sum()
    print(f"Total duration: {total_seconds/3600:.2f} hours ({total_seconds:,.2f} seconds)")
    print(f"Mean: {df_female_duration['duration_seconds'].mean():.2f}s")
    print(f"Median: {df_female_duration['duration_seconds'].median():.2f}s")
    print(f"Min: {df_female_duration['duration_seconds'].min():.2f}s")
    print(f"Max: {df_female_duration['duration_seconds'].max():.2f}s")
    display(df_female_duration)

if female_audio_errors:
    print(f"\nErrors reading files: {len(female_audio_errors)}")
    display(pd.DataFrame(female_audio_errors))

Using folder: /Users/dmatekenya/Library/CloudStorage/GoogleDrive-dmatekenya@gmail.com/My Drive/Chichewa-ASR2/data/chichewa-speech-data/dev_test_aims_cameroon/Female
Exists: True

Audio files found: 742
Total duration: 1.22 hours (4,405.87 seconds)
Mean: 5.94s
Median: 4.88s
Min: 1.74s
Max: 27.65s


,audio_filename,duration_seconds
0,pepani.wav,27.65
1,mofaya_1.wav,27.50
2,munthu.wav,26.11
3,amama.wav,26.03
4,aud_from_vid19_1.wav,24.64
...,...,...
737,priscilla_self_records151.wav,2.07
738,AU39.wav,2.05
739,AU113.wav,1.81
740,AU122.wav,1.76


In [6]:
from transformers import pipeline

# Load model
clf = pipeline(
    "audio-classification",
    model="prithivMLmods/Common-Voice-Gender-Detection"
)

output_csv = DIR_DATA / "gender_predictions.csv"
threshold = 0.9  # confidence threshold for "female"

results = []
SKIP_KEYWORDS = {"long", "interview", "social"}

for fname in os.listdir(dev_test_dir):
    if fname.lower().endswith(('.wav', '.mp3', '.m4a', '.aac', '.ogg', '.flac', '.mp4', '.opus')):
        # Skip files with certain keywords
        if any(keyword in fname.lower() for keyword in SKIP_KEYWORDS):
            continue
        
        audio_path = dev_test_dir / fname
        try:
            result = clf(str(audio_path))[0]
            label = result['label']
            score = result['score']
            
            # Get duration from df_rich_ok
            row_match = df_rich_ok[df_rich_ok['audio_filename'] == fname]
            duration = row_match['duration_seconds'].values[0] if not row_match.empty else None
            
            results.append({
                "filename": fname,
                "label": label,
                "probability": round(score, 3),
                "duration_seconds": duration
            })
        except Exception as e:
            print(f"Error processing {fname}: {e}")

# Create DataFrame
df_gender = pd.DataFrame(results)



Loading weights: 100%|██████████| 215/215 [00:00<00:00, 9592.64it/s]


In [7]:
df_gender.head(10)

,filename,label,probability,duration_seconds
0,dunstan_self_recorded_256.wav,male,0.999,3.39
1,priscilla_self_records548.wav,female,0.996,6.41
2,asante_records_99.wav,male,0.999,4.76
3,WAU2110.wav,male,0.999,15.00
4,dunstan_self_recorded_242.wav,male,0.999,6.10
5,priscilla_self_records206.wav,female,0.864,4.16
6,priscilla_self_records560.wav,female,0.996,8.13
7,andrew_records_187.wav,male,0.999,20.71
8,andrew_records_193.wav,male,0.999,22.41
9,priscilla_self_records574.wav,female,0.955,3.18


In [15]:
speaker_hours.to_csv(DIR_DATA / "speaker_duration_summary.csv", index=False)    

In [ ]:
for file in os.listdir(audio_dir):
    if file.endswith((".wav", ".mp3", ".flac")):
        path = os.path.join(audio_dir, file)

        try:
            preds = clf(path)

            # Find female prediction
            female_score = 0.0
            for p in preds:
                if "female" in p["label"].lower():
                    female_score = p["score"]

            is_female = female_score >= threshold

            results.append({
                "file": file,
                "female_score": round(female_score, 3),
                "flag_female": is_female
            })

        except Exception as e:
            print(f"Error processing {file}: {e}")

# Save results
with open(output_csv, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["file", "female_score", "flag_female"])
    writer.writeheader()
    writer.writerows(results)

print(f"Saved results to {output_csv}")

In [11]:
df_rich_ok.head(10)

,audio_filename,transcript_filename,duration_seconds,duration_bucket,transcript_word_count,transcript_char_count,words_per_second,characters_per_second,speaker_name,speaker_sex,num_speakers,review_flag
0,220422-123447_nya_e46.wav,220422-123447_nya_e46.txt,5.97,2,5,36,0.84,6.03,dunstan,Male,1,ok
1,220422-135445_nya_e46.wav,220422-135445_nya_e46.txt,6.96,2,7,51,1.01,7.33,dunstan,Male,1,ok
2,220422-135511_nya_e46.wav,220422-135511_nya_e46.txt,9.98,2,11,77,1.10,7.72,dunstan,Male,1,ok
3,220422-151734_nya_e46.wav,220422-151734_nya_e46.txt,12.37,3,17,124,1.37,10.02,dunstan,Male,1,ok
4,220422-151915_nya_e46.wav,220422-151915_nya_e46.txt,11.73,3,17,103,1.45,8.78,dunstan,Male,1,ok
5,220422-151957_nya_e46.wav,220422-151957_nya_e46.txt,9.96,2,12,77,1.20,7.73,dunstan,Male,1,ok
6,220422-152145_nya_e46.wav,220422-152145_nya_e46.txt,5.85,2,6,47,1.03,8.03,dunstan,Male,1,ok
7,220422-152224_nya_e46.wav,220422-152224_nya_e46.txt,9.48,2,10,68,1.05,7.17,dunstan,Male,1,ok
8,220422-152257_nya_e46.wav,220422-152257_nya_e46.txt,11.12,3,11,86,0.99,7.73,dunstan,Male,1,ok
9,220422-152711_nya_e46.wav,220422-152711_nya_e46.txt,6.62,2,7,43,1.06,6.50,dunstan,Male,1,ok


In [9]:
df_filenames = pd.read_csv("~/Downloads/dev_filenames.txt", header=None, names=["audio_filename"])

In [10]:
df_filenames.head(10)

,audio_filename
0,LongAudio150.wav
1,asante_records_328.wav
2,transcribed_social_research_interviews_5201.wav
3,transcribed_social_research_interviews_5281.wav
4,andrew_records_160.wav
5,transcribed_social_research_interviews_6121.wav
6,transcribed_social_research_interviews_793.wav
7,LongAudio187.wav
8,transcribed_social_research_interviews_5239.wav
9,transcribed_social_research_interviews_436.wav


In [11]:
# Define source and destination directories
source_dir = DIR_DATA_GOOGLE_DRIVE  # This is DIR_GOOGLE_SPEECH
dest_dir = DIR_DATA / "dev"
dest_dir.mkdir(parents=True, exist_ok=True)

transcripts = []
copied = 0
missing_audio = []
missing_txt = []

for idx, row in df_filenames.iterrows():
    audio_fname = row["audio_filename"]
    # Search for audio file recursively
    found_audio = None
    for f in source_dir.rglob(audio_fname):
        if f.is_file():
            found_audio = f
            break

    if found_audio:
        dest_audio = dest_dir / audio_fname
        if not dest_audio.exists():
            shutil.copy2(found_audio, dest_audio)
            copied += 1
        # Look for transcript (.txt with same stem)
        transcript_path = found_audio.with_suffix(".txt")
        if transcript_path.exists():
            try:
                transcript_text = transcript_path.read_text(encoding="utf-8", errors="replace").strip()
            except Exception:
                transcript_text = None
                missing_txt.append(audio_fname)
        else:
            transcript_text = None
            missing_txt.append(audio_fname)
    else:
        missing_audio.append(audio_fname)
        transcript_text = None

    transcripts.append(transcript_text)

df_filenames["transcript"] = transcripts

print(f"Copied {copied} audio files to {dest_dir}")
print(f"Missing audio files: {len(missing_audio)}")
print(f"Missing transcripts: {len(missing_txt)}")
df_filenames.head(10)

Copied 1005 audio files to /Users/dmatekenya/git-repos/chichewa-asr/data/dev
Missing audio files: 0
Missing transcripts: 61


,audio_filename,transcript
0,LongAudio150.wav,"zi mukhoza kukatenga mafuta yanyama mukaphika,..."
1,asante_records_328.wav,Ndibweretsa ndingomaliza zimene ndikupangila a...
2,transcribed_social_research_interviews_5201.wav,Ndiyeno a mpingo aja amapitako kumakaona kumak...
3,transcribed_social_research_interviews_5281.wav,NaN
4,andrew_records_160.wav,Moti kukanako ndi mantha osati kuti zimene ama...
5,transcribed_social_research_interviews_6121.wav,pa mwezi paja ndi kawiri kamodzi basi
6,transcribed_social_research_interviews_793.wav,nthawi ya 2 ocklock\nUmmhum\nYah! Kuchipatalak...
7,LongAudio187.wav,mukuona ngati chomwe chapangitsa kusintha kume...
8,transcribed_social_research_interviews_5239.wav,Mmene angauzire Mulungu kuti tiyenera kupemphe...
9,transcribed_social_research_interviews_436.wav,kayambe ya kachaso bwanji?\nNdi mbali yanga; z...


In [12]:
from pydub import AudioSegment

durations = []
for fname in df_filenames["audio_filename"]:
    audio_path = dest_dir / fname
    if audio_path.exists():
        try:
            audio = AudioSegment.from_file(audio_path)
            duration = len(audio) / 1000.0  # seconds
        except Exception:
            duration = None
    else:
        duration = None
    durations.append(duration)

df_filenames["duration_seconds"] = durations
df_filenames.head()

,audio_filename,transcript,duration_seconds
0,LongAudio150.wav,"zi mukhoza kukatenga mafuta yanyama mukaphika,...",15.000
1,asante_records_328.wav,Ndibweretsa ndingomaliza zimene ndikupangila a...,5.921
2,transcribed_social_research_interviews_5201.wav,Ndiyeno a mpingo aja amapitako kumakaona kumak...,15.000
3,transcribed_social_research_interviews_5281.wav,NaN,15.000
4,andrew_records_160.wav,Moti kukanako ndi mantha osati kuti zimene ama...,26.149


In [41]:
search_folder = DIR_DATA_GOOGLE_DRIVE
dest_dir = DIR_DATA_GOOGLE_DRIVE / "consolidated-subset-with-transcripts/dev2"
test_dir = DIR_DATA / "test"
df_transcript1 = pd.read_csv(DIR_DATA_GOOGLE_DRIVE / "transcripts.csv")
df_transcript2 = pd.read_csv(DIR_DATA_GOOGLE_DRIVE / "dev_test_aims_cameroon/metadata_transcripts.csv")
df_transcript3 = pd.read_csv(DIR_DATA_GOOGLE_DRIVE / "transcript-additioonal-audios.csv")
df_metadata_test = pd.read_csv(DIR_DATA / "test/metadata.csv")


In [42]:
txt_files_lookup = {}
for txt_path in DIR_DATA_GOOGLE_DRIVE.rglob("*.txt"):
    txt_files_lookup[txt_path.name.lower()] = txt_path

In [31]:
dev_filenames_set = set(df_metadata_test["audio_filename"].str.lower())

In [34]:
from pathlib import Path

# Build set of filenames already in dev (from df_metadata_test)
dev_filenames_set = set(df_metadata_test["audio_filename"].str.lower())

# Build transcript lookup from .txt files in DIR_DATA_GOOGLE_DRIVE
txt_files_lookup = {}
for txt_path in DIR_DATA_GOOGLE_DRIVE.rglob("*.txt"):
    txt_files_lookup[txt_path.name.lower()] = txt_path

# Helper: search transcript in all transcript dataframes
def find_transcript_in_dfs(audio_fname):
    stem = Path(audio_fname).stem.lower()
    # df_transcript1: columns ["Audio file name", "Text file name", "Text"]
    if 'df_transcript1' in globals():
        match = df_transcript1[df_transcript1["Audio file name"].str.lower() == audio_fname.lower()]
        if not match.empty:
            return match.iloc[0]["Text file name"], "df_transcript1", match.iloc[0]["Text"]
    # df_transcript2: columns ["filename", "chichewa-transcript"]
    if 'df_transcript2' in globals():
        match = df_transcript2[df_transcript2["filename"].str.lower() == audio_fname.lower()]
        if not match.empty:
            return np.nan, "df_transcript2", match.iloc[0]["chichewa-transcript"]
    # df_transcript3: columns ["audio_fname", "transcript"]
    if 'df_transcript3' in globals():
        match = df_transcript3[df_transcript3["audio_fname"].str.lower() == audio_fname.lower()]
        if not match.empty:
            return np.nan, "df_transcript3", match.iloc[0]["transcript"]
    return None, None, None

records = []
copied = 0

for audio_file in DIR_DATA_GOOGLE_DRIVE.rglob("*"):
    if not audio_file.is_file() or audio_file.suffix.lower() not in {".wav", ".mp3", ".m4a", ".aac", ".ogg", ".flac", ".mp4", ".opus"}:
        continue
    audio_fname = audio_file.name
    # Omit if already in dev
    if audio_fname.lower() in dev_filenames_set:
        continue

    # Check duration and skip if >35 seconds
    try:
        audio = AudioSegment.from_file(audio_file)
        duration = len(audio) / 1000.0
    except Exception:
        duration = None
    if duration is None or duration > 35:
        continue

    # Transcript: check for .txt file
    transcript_fname = f"{Path(audio_fname).stem}.txt"
    transcript_path = txt_files_lookup.get(transcript_fname.lower())
    transcript_source = None
    transcript_text = None
    if transcript_path and transcript_path.exists():
        try:
            transcript_text = transcript_path.read_text(encoding="utf-8", errors="replace").strip()
            transcript_source = "txt_file"
        except Exception:
            transcript_text = None
            transcript_source = "txt_file_error"
    else:
        # Try transcript dataframes
        t_fname, t_source, t_text = find_transcript_in_dfs(audio_fname)
        if t_source:
            transcript_fname = t_fname if pd.notna(t_fname) else transcript_fname
            transcript_source = t_source
            transcript_text = t_text

    # Only copy if transcript exists
    if transcript_text:
        dest_audio = dest_dir / audio_fname
        if not dest_audio.exists():
            shutil.copy2(audio_file, dest_audio)
            copied += 1
        records.append({
            "audio_fname": audio_fname,
            "transcript_fname": transcript_fname,
            "transcript_source": transcript_source,
            "transcript": transcript_text,
            "duration": duration
        })


In [ ]:
df_metadata_dev = pd.DataFrame(records)
# Also skip files with "priscilla" in the filename
df_metadata_dev = df_metadata_dev[~df_metadata_dev["audio_fname"].str.lower().str.contains("priscilla")]
df_metadata_dev.head()

In [50]:
audio_fname_list = [f.name for f in dest_dir.iterdir() if f.is_file()]
audio_fname_list[:10]  # Show first 10 as a sample

['dunstan_self_recorded_256.wav',
 'transcribed_social_research_interviews_510.wav',
 'transcribed_social_research_interviews_276.wav',
 'priscilla_self_records548.wav',
 'asante_records_99.wav',
 'transcribed_social_research_interviews_262.wav',
 'dunstan_self_recorded_242.wav',
 'transcribed_social_research_interviews_7195.wav',
 'andrew_records_187.wav',
 'andrew_records_193.wav']

In [52]:
# Remove file extensions and compare stems (case-insensitive)
audio_stems = {Path(f).stem.lower() for f in audio_fname_list}
txt_stems = {Path(f).stem.lower() for f in txt_files_lookup.keys()}

matched_stems = audio_stems & txt_stems

# Get the original audio file names that have a matching transcript
matched_audio_files = [f for f in audio_fname_list if Path(f).stem.lower() in matched_stems]

print(f"Matched audio files with transcripts: {len(matched_audio_files)}")
matched_audio_files[:10]  # Show a sample

Matched audio files with transcripts: 4207


['dunstan_self_recorded_256.wav',
 'transcribed_social_research_interviews_510.wav',
 'transcribed_social_research_interviews_276.wav',
 'priscilla_self_records548.wav',
 'asante_records_99.wav',
 'transcribed_social_research_interviews_262.wav',
 'dunstan_self_recorded_242.wav',
 'transcribed_social_research_interviews_7195.wav',
 'andrew_records_187.wav',
 'andrew_records_193.wav']

In [53]:
from pathlib import Path

# Remove file extensions and compare stems (case-insensitive)
audio_stems = {Path(f).stem.lower() for f in audio_fname_list}
txt_stems = {Path(f).stem.lower() for f in txt_files_lookup.keys()}

matched_stems = audio_stems & txt_stems

# Get the original audio file names in dest_dir that have a matching transcript
matched_audio_files = [f for f in audio_fname_list if Path(f).stem.lower() in matched_stems]

print(f"Matched audio files with transcripts: {len(matched_audio_files)}")
print(matched_audio_files[:10])  # Show a sample

# Optionally, build a records list with transcript text and duration
from pydub import AudioSegment

records = []
for fname in matched_audio_files:
    audio_path = dest_dir / fname
    txt_path = txt_files_lookup.get(f"{Path(fname).stem.lower()}.txt")
    try:
        transcript = txt_path.read_text(encoding="utf-8", errors="replace").strip() if txt_path and txt_path.exists() else None
    except Exception:
        transcript = None
    try:
        duration = len(AudioSegment.from_file(audio_path)) / 1000.0 if audio_path.exists() else None
    except Exception:
        duration = None
    if transcript:
        records.append({
            "audio_fname": fname,
            "duration": duration,
            "transcript": transcript
        })


Matched audio files with transcripts: 4207
['dunstan_self_recorded_256.wav', 'transcribed_social_research_interviews_510.wav', 'transcribed_social_research_interviews_276.wav', 'priscilla_self_records548.wav', 'asante_records_99.wav', 'transcribed_social_research_interviews_262.wav', 'dunstan_self_recorded_242.wav', 'transcribed_social_research_interviews_7195.wav', 'andrew_records_187.wav', 'andrew_records_193.wav']


In [54]:
df_metadata_dev = pd.DataFrame(records)
df_metadata_dev.head()

,audio_fname,duration,transcript
0,dunstan_self_recorded_256.wav,3.392,Ku Malawi kuno mpaka a chifwamba eeti.
1,transcribed_social_research_interviews_510.wav,15.000,"Aaah, Inu kaya muna chiyambileni kale mwakhala..."
2,transcribed_social_research_interviews_276.wav,15.000,ogulitsa mwina ma groceries wo wina ndi wa bu...
3,priscilla_self_records548.wav,6.409,Kodi iwe upita nawo mu mtsinje ifetu ndiye tik...
4,asante_records_99.wav,4.760,Ngati samwalira tigwira tikaika tokha sanganyo...


In [58]:
df_metadata_dev.shape  # Count how many transcripts are missing (NaN)

(4207, 3)

In [60]:
def get_transcript_from_dfs(audio_fname):
    stem = Path(audio_fname).stem.lower()
    # df_transcript1: columns ["Audio file name", "Text file name", "Text"]
    if 'df_transcript1' in globals():
        match = df_transcript1[df_transcript1["Audio file name"].str.lower() == audio_fname.lower()]
        if not match.empty:
            return match.iloc[0]["Text"]
    # df_transcript2: columns ["filename", "chichewa-transcript"]
    if 'df_transcript2' in globals():
        match = df_transcript2[df_transcript2["filename"].str.lower() == audio_fname.lower()]
        if not match.empty:
            return match.iloc[0]["chichewa-transcript"]
    # df_transcript3: columns ["audio_fname", "transcript"]
    if 'df_transcript3' in globals():
        match = df_transcript3[df_transcript3["audio_fname"].str.lower() == audio_fname.lower()]
        if not match.empty:
            return match.iloc[0]["transcript"]
    return None

copied_records = []
for audio_fname in audio_names:
    fname, duration = copy_audio_if_short(audio_fname, search_dirs, dest_dir)
    if fname and duration is not None:
        transcript = get_transcript_from_dfs(fname)
        copied_records.append({
            "audio_fname": fname,
            "duration": duration,
            "transcript": transcript
        })

df_metadata_dev2 = pd.DataFrame(copied_records)
df_metadata_dev2.head()


,audio_fname,duration,transcript
0,WAU52.wav,12.564,zazo zikhala zokonza chaka cha mawa. amanena z...
1,WAU38.wav,28.556,potindi chaka cha mawa. Tikhoza kuuika mwina p...
2,WAU36.wav,22.655,"munthu ndi munthube, olo titanena 100 tipezeka..."
3,WAU120.wav,16.981,Ndalumikizana ndi Gloria apapa. Ndimafunsa kut...
4,WAU33.wav,29.977,Panopa nthumba la simenti sindikudziwa kuti zi...


In [73]:
df_metadata = pd.concat([df_metadata_dev, df_metadata_dev2], ignore_index=True)

In [63]:
df_metadata.duration.sum() / 3600  # Total hours of audio in the dev set

np.float64(14.349410833333334)

In [65]:
df_metadata.to_csv(dest_dir / "metadata.csv", index=False)

In [66]:
# Remove all files with "priscilla" in the filename from dest_dir
for f in dest_dir.iterdir():
    if f.is_file() and "priscilla" in f.name.lower():
        f.unlink()

# Remove from df_metadata any rows where audio_fname contains "priscilla"
df_metadata = df_metadata[~df_metadata["audio_fname"].str.lower().str.contains("priscilla")].reset_index(drop=True)

# Ensure none of the files in dest_dir are present in test_dir
test_filenames = {f.name.lower() for f in test_dir.iterdir() if f.is_file()}
for f in dest_dir.iterdir():
    if f.is_file() and f.name.lower() in test_filenames:
        f.unlink()
        # Also remove from df_metadata if present
        df_metadata = df_metadata[df_metadata["audio_fname"].str.lower() != f.name.lower()].reset_index(drop=True)

In [68]:
df_metadata.to_csv(dest_dir / "metadata.csv", index=False)

In [75]:
df_metadata.duration.sum() / 3600  # Total hours of audio in the dev set

np.float64(14.349410833333334)

In [ ]:
# Get set of audio filenames actually present in dest_dir
present_files = {f.name for f in dest_dir.iterdir() if f.is_file()}

# Filter df_metadata to only rows where audio_fname is present in dest_dir
df_metadata_synced = df_metadata[df_metadata["audio_fname"].isin(present_files)].reset_index(drop=True)

print(f"Rows before sync: {len(df_metadata)}")
print(f"Rows after sync: {len(df_metadata_synced)}")

df_metadata = df_metadata_synced
df_metadata.head()

Audio files with transcript in df_metadata: 0 / 3919


""


In [ ]:
df_

In [37]:
from pydub import AudioSegment

audio_exts = {'.wav', '.mp3', '.m4a', '.aac', '.ogg', '.flac', '.mp4', '.opus'}
total_duration = 0
audio_files = [f for f in dest_dir.iterdir() if f.suffix.lower() in audio_exts]

for audio_file in audio_files:
    try:
        audio = AudioSegment.from_file(audio_file)
        total_duration += len(audio) / 1000.0  # seconds
    except Exception as e:
        print(f"Error reading {audio_file.name}: {e}")

print(f"Total duration in {dest_dir}: {total_duration/3600:.2f} hours ({total_duration:,.0f} seconds)")
print(f"Total audio files: {len(audio_files)}")

Total duration in /Users/dmatekenya/Library/CloudStorage/GoogleDrive-dmatekenya@gmail.com/My Drive/Chichewa-ASR2/data/chichewa-speech-data/consolidated-subset-with-transcripts/dev2: 14.53 hours (52,293 seconds)
Total audio files: 4273


In [36]:
df_metadata_dev.duration.sum()/3600

np.float64(127.82183611111111)

In [10]:
list(df_rich.speaker_name.unique())

NameError: name 'df_rich' is not defined

In [53]:
df_consolidated_updated.duration_seconds.describe()

count    4650.000000
mean       14.942503
std        65.724757
min         0.060000
25%         6.080000
50%        15.000000
75%        15.000000
max      1849.680000
Name: duration_seconds, dtype: float64

In [9]:
f_consolidated_updated.head(10)

NameError: name 'f_consolidated_updated' is not defined

In [ ]:
from src.utils import load_audio_duration_cache, save_audio_duration_cache